# GeoLifeCLEF v31 — full-data bagging + environmental analogues

**An unscored candidate, not a demonstrated SOTA result.** V30 regressed to
0.23451 public / 0.20942 private. V29 remains the best: 0.23900 / 0.21093.

This version freezes the exact scored v29 CSV, preserves **every row's species count**
and its leading 60%, and allows at most 2 or 4 tail replacements. It adds independently
seeded replicas of the three v29 neural models and an environmental-neighbour expert.
Production uses **all 88,987 PA surveys**, with no v30 calibration-anchor exclusion.
The habitat metric uses official environmental features, not coordinates, country or IDs.

## Run and submit

1. Attach **GeoLifeCLEF25 @ CVPR & LifeCLEF** (`geolifeclef-2025`). No extra dataset.
2. Select **GPU T4 x1**, restart the session, then **Run All**. Internet is not needed.
3. Submit **only** `v31_export/GLC25_PA_submission_v31.csv`, and only if the final
   `eligible_for_submission` is `true`. Never submit the ZIP, reports, diagnostic CSV,
   NPZ, or anything named `DO_NOT_SUBMIT`.

If a gate fails, the notebook returns the unchanged best-v29 prediction file explicitly
marked **DO_NOT_SUBMIT**, avoiding another identical submission.

## Evidence and limits

The 15 ranking policies are selected on calibration partitions, then checked on the
same two geographically buffered development folds as v30. Positive pooled gain is
insufficient: both folds, equal-country macro gain (countries with >=30 surveys),
gain outside Denmark/Netherlands, and the spatial-bootstrap lower bound must pass.
**All PA IDs were assessed in earlier experiments: these are repeated checks, not a
fresh independent audit.** Geographic gates are a response to v30 diagnostics, so they
too are development choices. The reference is a fixed-epoch v29 recipe refit; only
the official-test control CSV is byte-exact. Production replicas reuse the scored
v29 calibration coefficients across seeds, an explicit transfer assumption.

The original v29 production epoch schedule is frozen at 12/15/18 for both development
groups and production. There are 12 development fits and at most 3 production fits.
Whole-production admission is based on measured development speed. A **10.75-hour
cooperative guard** leaves headroom within the requested 12-hour session, but full
v31 T4 runtime is unmeasured: a slow run may stop safely instead of completing.

The successful export contains exactly five files, capped at **16 MB total**. Large
temporary caches and checkpoints are removed. No external data or pretrained weights.


In [ ]:
"""v31: full-data seed bagging and habitat analogues, with cardinality-locked residuals.

Repeated development evidence only: every official PA ID has already been assessed.
The scored v29 CSV remains immutable; no result here establishes SOTA.
"""
from __future__ import annotations

import base64
import gc
import hashlib
import json
import lzma
import math
import os
from pathlib import Path
import traceback

import numpy as np
import pandas as pd
import torch
from scipy import sparse
from sklearn.decomposition import PCA

import types as _types
_v27_source = lzma.decompress(base64.b64decode('...'))
assert hashlib.sha256(_v27_source).hexdigest() == '69a2d3952fa6324d9a05f4edb6fa7662388a198df89c407f7f034b94099f1dc0'
_v27 = _types.ModuleType('v31_frozen_v27')
exec(compile(_v27_source, 'frozen-v27', 'exec'), _v27.__dict__)
del _v27_source
_v29_source = lzma.decompress(base64.b64decode('/Td6WFoAAATm1rRGAgAhARwAAAAQz1jM4LmJNRhdABFoGYZDkepR8662PQUn4xZ6XcmQ/hL/kvr9CZNu7OZ7BTsxRAsOYMBV+kEfAlVaRLuDeekbHjOtmuhTxBdCOe6HZJxwqqDt9qVrPyu5X/6NGy0QdJbngcotqULQFfacScBNbbdOSStqetO64QR/vO+S7bUNqUujgjGz6OwXn8MwHMXx8+dfACclTWCPZVIHtlChHwzHKt6Y4ruUbOhXdtRYx9dJxcG0YkGGDH3GOx26MR+2b4jmjGZJgwPkbQZkg9SQnr4jhhslVFkUbM5afMmX7W0IB6Ewk/S5PZgJ1SC23uplBm9knz11eO7XQAi3M5vScHP9bMPRmdQmfc8LLFNU5urMKu8F7fxjOfqWz61KT7p/B7DyxqthdRmp2OS49OYZjVzNtiUHL4MBBrjuQWPOa/mduQ0BKN7Zv4ZRQwM/ufnBCTBREOaEMmbxG8PQxQH++E5A9GlVRengBZdVr5BibVTWhP8NIL2mNSGSft2Z8wKd1it0TdhTmiW7LduiasxqQyMt6ORk5QdGzdpkhnHPDQrQ2ZR9K1gzycBeAGInDTh5TmGesWv5U4WFhVXbDj96/SELUnFgscDQo9FhX32osEXOTrlFLTKorQ5RUisOb229ARqdhRDkYgVsFuxJ3cEtLOab7wwVhAKY9teIDIB+wGNKtSDNoRHSlfA5bbBJSHws1IpXfL7PfoaVX6gg1NZj80nzWcK+1md1+jdo30JOx9ocDo7t4evgHZjqMOiOhHfWzCRapVTJ+DaU+/IcoIxWQmqH3mvSwW9FcxwK0KSV8I1D2+94pR4feMHmmZYNfu5TEuSiDw5wAxoZzwsed0hyv3rx9JmL+brlPL1PwRW9vsy/ysc3ebWwish0hRNdl9cDs3Dx/G/bCVU8i/68DhYJNbTWJvVjA7j+waw4s5Zks/1OrIuiDIhx0T68q645m0Djti81Ezr5L6UqCq2M7iNAawqJpbU87D9ozuYun5J0wTGdf8FvjxbvzXBxQPXPxlsYYGiiTEakM2Xqhw55lk9f8r5CGh2DaisljGM39V2+/gmaFVDxFMuF2dCu2eVgEpBimQC9D7t5BsUqj6r30Sh6VZmNZzCq6r5wGXOpps7oYCicggl71E5uD85L767gtSZHn3s2TqKmoRyJaoHFfxvalcV3baX0ng3XPi5Snaivi0KK/K9cJVzUUhZGdFZ70Bj2uDsOOlIevgVk6GuGln6EFr0Uq8re9hh6SbVcW51ODb+2EvVWa0c1bNUntNNKX/Q1FqhKzny67Nzu2IDaiCz3tIgp8zp1ASB6/NHvkNHpxpLxl0pXkcWq+JeVyam3ZLYE3zZeFiT/o+wW67F5h8uGookMexDdyiVxvQnG7x13/Xl2zIRNqG6PWEy58VzfQgD7uQ1JVcjMocJwJZChJ4Fcc7L7LBf0O09Zrua3jHgdX7pfHq1URJempsEjb+abyJWglpIZvKD37oB1+l+rjgyY1tuMq+CKKw2BaAk6WnW6zcQBKAt5EvNMUBolL4Ou29La0McA8XT5H+8yVeObyLPUayQ2oMJVcREItshLXAmKRcM6pawihm7fj7Djqhcj1cUL/2IJdZuxlfF8t7KIcWeXCvWppm5Pe+W1T31EXoCqulGa2fO5t+ERM+uRefl8nUx/iTHtSJjENjfAryvjH668TftvvCPOVqWp8cqr0CJaETBe9ye3iT5YXbNHwx9WRgcfbmA71iyNbGTtIzINlnY4pol9FrZEfSkSQSL+RDZdkrVrCJO2SjOQxbSOlfmhAais47Bax2IiYmOpRx2/0IIeb51ZPcumd/P0+cjJesRtnPaUcLSiK3Lrrr9x0trgUoXfixns9vlLUrYVvPv+PQngBXQmdhMkxjan6Ic8boh82mpOAa2FYnXaxQ/aDvZVx/UiQIFR1okDsFpEMLZo2tLMpv30Hb/bbUDjAB3ufU5EkW2MxU4pyxasVxmpl0C8sGAHY4WHOhX2W8jxG33d3N7WqQoGy17tnPEtkyuvz9IXcoFxFxyg3zo017B8L57TCYKPN/UPJBr6974K86LPkwLU2mtNtm4pXxsDmguv+JtumwjK2WD02J1qfyI1kYfkN5kxYfbCB2ZjmSD0ZtyyB/1zdqPHyAQKjgVRLBt+R6MtQT0ek9LSZ6TWswg2wkR1JCxhJuSLhANQOWznTaf3oLbljc49U4z419PZkkegH433FRGJkpf6UwEmEzDIL3qoxAX8UOqzTvibEJOBpCFPkZ0IAacKQ1VYg9gz+Z+BTmh2NDRmcFtTDOLqSpJpJIXIpxWhPFQ5cMamVaBY34rucBB3tAMYWJKFl5kl5tmbDpYnOLxGHdY2WAybjidDJ4Vkq2o83MP3vmLzKz6MfoKgtsv3tw4BeiUcrXLUs93TCpUxjSiSXwP130mnu8d4gE0Zp8B87qd5BQfXRO+ml8s+QU+faCl8xvEWjkCYlZA6QKQd+OaHfrDD6cdi5H66CPmjHUH+kTw4CpXpYfcg3o3EFKp+hfHpGVb7TVVxbI16bV50Wu7kUIZv/W2S1JtUGhiF4xuqasLLXW+ngUaAt3IdZS6nJUd4qZXbJ/gdp1a9vZ9vhBZcAVgiTJSyt4hZw4Ze6syfZws1Ice5KeRvLKVT+m+LJv8d33W5NRPyb/E1SVOiV7bZqQVOQCm096Fm23/3OsX5wX3bR6XMCoL86ORqEnj6oPkgTC6z6JJjKuLTsKjV8u9L/8mhXxECLUCHtOnf++EZwkt0uVAQWagr8sXkiQWya9xz/3CgaE15i5r3EjUrJxdjrEobvcx1biGYPwEwUhpDdEL+uoe+2WYAIDLYODNDRy/X/PclMR/hk9OfmeZuh6gCZgAyqR0fc6CnnMjkF27iXOmEVL7nKtEBch5dVZnT+17dY45WuStXagSuvwyGx9/Qth9AUVN78rGwKykqUbKXvAqG6KxVxfIZNBqt/mvNxbSCgOJBCbsAw48ZrmumDnJXS0hNEL1EPpvCklDGp1mvd/S/arZFfWlmNasmNOJMIUWLZPjgtbgMa8n7LzDqvBKGY7+5LqFb+CCifezZQf/fBaO/isvk8XbFn4zg/g9JwYirWS+BtkrU23RgzvsG13Zcc/VwpIZT1Q/W6YuPROm4YxkBM7BmX6U7mFyUQ/FTXdptZF9RZYryb/6ro2rtl6hqqlTj40R70MowghC+1Ru/QF2VW787hzdJteyfKpu1+sVQJ+ecnIXkYr+UGOyeF1cY5cxdF8uHJQOGKfgpANzMkvd7BLpGY2hnBoujda7Tw0rHtCRzIqNXJm+l0YEc4yPbd4jVWN+x6m+UKXBV+K8ryGUjFai9N7bzaIdMqixHThGULLw3RLIe1TEBMbbXcM/m3flvybkc+1Q7W84JOdCbhS1xA9uB6ATYD8bIjODXea5h3J4oXRZGUCgjo3voD5WcfT0Z2N76Wp7yFyJouSMG4b+ns3dL92hQ4l71n8tK01zjP7woMAgAft5LapHkqM4xoqKp4dFhcPaz/GNaSVzloNFFAFwbwYX698dVPD52ur0JFSQ5bfVhq2fnU/pvWBC0YBHJxYKA7mLxZjyGpxoyXxZI0whEs6QW4WALJvlj6Ui7iJEHHTTW+K2pgl2qb6hG3U1ZjS679aEcSzyRVqEbTSWbOqRqXAOy8IRqU03BoS/tRzlkM3LbwG3kR24RlghamLaU0GC19N8M90BAtLpQ1EsE6irWF4k8YZAzG42pIaLe6hjLQ3xzisxhdv3gEQFFkXwmLNIUq3FUOlV4I34em5praKrKPM/JrSALvwg2pMiduYim8oM3oXLUVWrsFfd48wIlNnOjOmdtvUQcuBMqTpspxwYz9qJKJemD9Vd6q/bZP2QvjBfkJlJOSIjAkQJBTdvZCTUt52EqKBqjfaqxTsW/TgazT329x4ExqMUX+3wZ4Yjr0WpQ/LHuY2a1ZHt5UZ/+bA7+KS5gJswKoTMZkJ6TVxYERRLbzlBtY4sNMj+WK63qWOJT5JNzhxdb66u6a3bNcsiBQKbcqNevLvxhEYYLn1c1rgTk1uGZIOb9KkoNNIkMP8PaEwafUt5Cmso0co5FNq86E8upnNuTvz+LIBpUcD381dnzVNR99IjHu8q59Hx7/LySL1IrQfraVX5zu9Um0n5OXlsr87bd6rDodr3WwQaHR40W65jnjy1z04aLy4qq3zpfP52vR4vuCl/9oVmEUTLpVWU12xkXrFmVRaEHhi9kv5s4FM+Y3tVw4aI+95xFaLB0sADddR8Xw6BFMMXm4sb0vZ6w6Ccj1JspvYEngkw5eD9VQiDlFgIWf079NvXlQp8lysUcs29Q5NF5yahhdsjgqulo/kheFSGlT5ub4YYOZpPhcCr4ymgklLcdM6kuyHHwae/Z0NMtU4tqlx6gZXF1mYqH58m+irDPTtWS7jbv11YE4y+KMRWSthPUNGqTIlmySiK1r2DyGJ9dXndjX81RoLT14iGByACxg9fl78XdCPGwnp8DU8z/62GIFfQOOrRfDx959peHEsDym4ve/DkSREy16t4NyTmzb92j04lR6biC4NfOGK3ILWZJUkAvE/ZjW31ACRkQuCKNtUFMctbfjtgqkHBW223T8Ij94NSvn3PmUmmVIijVnlMleVPZrsLRi8s8IeNLjXbZ5xkOmjB3dkbH8763usD0JaUNuF4m6KMfsx833nTg4xCOliTLQqyVKWCOEWjPILk70SVMaDjofF8AmYEzjxgc2+NJqmJkuZlibwYAQZzUNqiZ1Dc/JeD0F7OxCNvnnZgMqD80QwXzcY2xy70QUu+zF/cKEvtwYibMVHoqxsVQDGF8/A4SmnFOG6D/VKVWvFsK2ptmgWPdC0jGBMv3r3BeDBxCLmQQ+W9rdRXBw6PEi0JX/520Mea6zatnhf0ojTSavySPNHpOisDVklNJw9rBZc/Ex5KixqHvVKS8agGEhk8fYZ/eBo8bJ7tu9YwBu6I+4EoNYoFbUFL4s7JnzYbc7MrJoP8wnSLDcU7Vs5ctvxrCi6FD1deyPD3Xw7sdcYqwr3Bdn5syX+2WiNJEAOgnS51Byz1awQR85oGef1iNHVH29PzACa9IbgPwjBLFPO8Jhh+WiwFcVbTNthmREisd2IgSSncwshwxlaTW/UUI9GCKPEA8NSi67zXYQcH1LDdv0j4ehho3ZuhfzDXV+L1cDinKVh48AxQ93u9OEbCE7Y4+2Mqx7C4pM9hvPmri5mYDOWaaReavsu552h6zP0DaX0lIRSxdSe2pQEh7Xd38DeoqbeYsL3SG68AYopSF3Pcy2fR4AEZqZYirhML7R1/RscZcxCi/don9VYu8swxsvkppi1d8FrdHjTxLRvbIhJeLe3KFj6yPouoUUShma0ZaicX8ASAxS6Ia5dy8lP5DK0TRFI9R8AXaTHlqW4do1YCvul+GgIE8mOm+vDvGPQ7me3U28ezKBBcRc4Jpdu8G9OCKuWXeObtnF7jya5+/hq0VSLZwBHGUJvTlqCAycZH9ijbC8U7XJuk5bbuCQ/rIL8ATDtUU4Tro22pemA6cJX6JRZYiocd1t6j6hslAt/bNoatgzNhRPdFiE05Y93OullpF1CujfD7U02uquUfqBfkBQKw3a1HRM3aiEB/Z5/8QgmIDO+sjI/lkBUTPAKX7uquN8blN4Na7gvlGbkvvY23csx8CNxsl+d3G1N4niYI59vUvDF4/caSsSnVotwdu1PIha1hOjsQe1YZU25KARG/OMy4w+o/8ir6BqXlB91hyjsIn/fzcJyN/S9xtMZmjeSjYco/T7Va9B9pSYRiB6wVdWyAHZ6TrnQI73aAYIs7ECG/5IQH3aYjaKm+WXmlnEV90S4sUCCoWh5dJRfB6GnfAk7QDK0d6h+HL6NWntkaHnt4C7z8hQIgNgdIQkMn0FGo2KtByHFYoMoz0aLD9+7zi7uw/huCFy2MYHk9md8nnW1mK2ERI1SDgFjcT8B5hKN+1bJCc5jP0WKahWK3W4wc3SC8VzS6Xy89A9AeOYKCaIARPaZPd/WOsHkTXaKgwChMGf7a34gUAgnZIie7RNgr07+8zapQ5h9C4iD9Ikh7fd+HvW75ji+gR+ejO2T3f79yoVbqG/tCofWzG27aEwc+O13k0nInNJO0z9gJGkdPZfwDEAlRYn4bi48CMIQDyL3miXjUu6XOpy9IJvc6tXcAMB+H3cHIwmQg8xETOVb0TsocSMsbVQHxIFcCYkGwmTToeCYSjfmMxe5XbDZn6XxH3Yzsbm4RDOma3xOFG8kfJ6DlSd8UUHu4jGhVsIBfEEOyOPDv3k2cZFmpgp2m6b82oXkFyklNqHgux8jjTSrwrsjVGMlDfttOCvNEe4+iY652n+zrN69oAw9+wqDnX2jx2zoWKFJTAztqu01XTFmHjGnBnpZCjt9Km5iyqqZVDmvIRN901TbhThJnn5+PrK9IVmctHQcqpWew2yEC3s5jKFtOC9zR8RS/Wjs59PF9DXOQFI8PCThIrWIhuyzmb1N2IgihT6IcL+fySQhztRn/Qdj9WEYVKq5i6tq1Tah3OVsWh+cPnm/kdeZr1lFRBjrUBhtaeFnfRvrYsDigJ7mDAImA0BS3ab7A1BQs6M9XBhcGi7BPfkN4CGPo98qyYcUkKExLUgeLsyuuud+wvKS3W8NE0wxyQNCCFEpV4ES/QNPiq8kvKdzZySGO7StlQRkdszMt1S+fhOTEkJ9P/RX40WyA8QMtjMBAJ+YbL611W5oV6aeiAuWa84/hVHiSwDYWEn9VF9PCHlw8wjFEGMxvEhW9/867/ULzObRqaUgbi6HiRFUtF7jR5p+/Xa+DND+vGk4O3VqOfS3+3P9Wa4tXVx7h9DJhspILxys9/+IVdcObJyWPVEe7ay7l4wn4NRKdS2PxaLnAbl60y0SKMxpplp5IIg+SxFjrT4Cy4Cjft4EF++n8HU4/1CU5Vo5j7iHuu5gDqa55gnY1wgurra/WO35F+OzC/dTYC33n8ZKmdf+9q9dBvpDtD/2N2BLdz4sSbqqw1kNdVjl166r+bOHPuNbSSgkpl5nwTE4rLMGEU29tvUTvi2M08jGo7Zkt1hZE/2JDmJONZkkyqMaWn2erNujDKnd9iF7z6lO1YtQZ1EG+k1kpK5O+rLOMYWgJBrMuAR+sirMfxCtUve67ukmo0QCj2O4yoXU+xRuKVj7SxGXxEJqeg4he2UR1MVLKDXVr1m2AcQl280Ma9vcKxlzz+BK95csZxzouMvYOtor3l1jQIIDX0+r2mM64QeoQouZskMGApq+CJnM8ABKnSCfCIHP2RSyFgveyxO1czE371k6W3gzYMke+EVXQ9IST8585WCB/3VAgw6L8dd7hFu0ZKYLxJ1HnFRORZ5bpNiDCbN1pKPevzMeiZUXoFeCMDuVWb/JRKEk1v/FI1ph1QVtgc8fnocg7F04woykcrM5bmFqRthm/QVoqY0O1j9LHV7c0xqo+ipVivoJHDf/OSUcKHowjnJQydWrpIw06thRyPONoirCB2aU+exwa4i62EruMoDBydieDk7OZ+I42Ps+VVG13I7PLoWK5FrmV+rrk8Y6+tNBNBNAjkHP5lpSK0yh1fcgvaHBsbi/VTJC/2GJcP+D+RGc7WOrBHmDM7fGG4x3IICl7EcVifPEL1lHCOee67rNrPnfwBQ6XyeOr7XTjZCEIEbS/VHtyt+tYIiDb/ykH06Ng+bD4e8Rz0ral6j8Dj2oufj1t0w9Skr9ihIES0wJ8Djw1q+nFDKhMeJOfen4bjAT/mvbtPWcIXXLXkTcRrk2HFF1V6cNOU/S/lL2kOOr4h1u8AKHw8oTuJ833iAKCXl09nTRemWKbvABzqF5kbtLwlcVvCn9zkmLqwi6fFmid6DobwcaY48dtiMG1mvlnXhBrKTf/f/4b6LX+TcefTiWyzGE5G5nhdrtTAbeyR9TtJzjU6mzay2ai36Or8e5CtFV1n44CPOiswLl/UtnCR7iXcm3jyRiOB2bCI2ra5UGVJXXVlg15nsEqI1uQTFm9Zqq8mnC7g7lFcsYVwry4qXD4hcAIwEHBRPq+r0DdI0l4L1pUH4f8+0xSBurl+cO3VG5sTprDYpZj1MMaoSlGIfE5c5iKTv8i8B2J8Q/4A2PkrFXxaw0E/iwk9l8B/3MEv7vSAHLEzUP8ANx5YT7rMJ1CNqIwWJUz3pinzMtglEYo/4vEVUKq/DJgmpZuwnuK9XuBJ3772PZGVh1pcPnp0cT27FPPsxSiAFn8PNdNfowYa0rR2BgX2Bmj1WPi/UVbATLE6Eit6KgP2NpuQDvoNAzvjUaU4+L0XhkCAWQoCLqH0+OzcBHZUjX0QzIZWKMrofdIm/Dp6QnLAsX3AEre+bjQBblZEi+PRRjA7XflFVWWyvf04jftv3AFivz6gCY3O5cV8UWPPGMzTbOgzAUYcxQmF87y/o8szaQ7bgjoR8qUtTDPwt/o2uo6UHmK/Wc4BRjgB2tDYyh6O8xNYvH7wqHdqTN+00aTEBIfFYIZGZ+/00rPpKButXY35t7+JawM6lw6ody9emDzeO+Xq0hvWY1KV95akemWbBzjlwsoH5noBHBqhPoBFeZ1mvI60bJU55pCkkiPAGuUSeQbozCOfr5lPbxWMufNMLUd80GMXZbNX3zJMTtXJSpw9Ij3iaMaJ7u4KJweHLSLMoSNrUImN5o/6g3Z/1PmHDdOF58Fmnxcy4Bquipf3RIAq0g91RpTDtH5JKiIsbpryn98FbXiZ5kXpozWoYlC9MzvcCBSgS1hFhhXwRSx55qyuI/S+kss8Vn1a1lLpREJAFHxw0GDhgDQwcvOfFevX93S971QTZ6mbWWtk51ZX9fKuMqXsSqyGT4XsGDT+v1IcTE6ByUL0gIR0QmQe/WZF9ECztIaGXFsvxr9/L1pvpfjeCbYNHgFPnKVyc0cntL3mEygRMBKLpO4n6qEPZ6/tpz6ThfMgZa7wz6Y2p3GqaHX98W3J6NPj4V1gw8jGLkYHry/nZ/H21B0qyfsXNCos05s9gMMT1OYKOxE8fZ3ZVnTfH9zmdr9k9LpXcAnaImY/jzUY3R86I4tz//QdI91IkzT26RwOpEv6DmLp3qMQB1i40bb10MZkbTfmucSPHuHHLS2jIsNRn0X8gygGDdNH5SG1jCBwRcbGVsMiPmj9aGiJE3DZjfqp5LFAjomNgZZ6GlSigYwA9OO+tGwE36bIMbuEdd2Yr6XcCIzRJvvrn5staakYympSS3+9Jn9bc1Vzi6zLoY/l02ABUv+ErlJijFWcYShfmLRssw84vXrG3S2eobl355RPwqFZKFAlDl68stV4UXE2xl64rm496SnHRsWb++5wDqBfAB1DH0P45DXVHfC+zAKk7CDudYoOBWJeCC3hPUiCGM+GNYXCGCfSs0kE2AtKUmz0LrBAnOBMbdD5/VormHo1QrtW4rwAtSlXFWxi8EeDDK/5uw6VBGE1/Te0GNZFHY9qo1UQpgnUz96ADSaavIQohAMC5z4INQ7gpqRKgwZSMZvpFMVXhBz7AyCY8snPsIh0HxLsZ9KCf46z1glDMtbiZZlMfEtljImOY8RAqO0rSyeGXf0nBZ6pJhkzlbiFbOoPM/xixg892SjG2uECJCHq+P+ZWTmYgnS1IrBLddnzEkKqM2b5NMyy4quhAAtfVre1ijNsRO9hPwmFXqFVIaLJbTfvfJIkosQIedCetx72xzNwPEte3NSdrswvUSgGJuXV6wpdaPHjmazSTAkLnxUCGdMrNMiIr0etPlta6xGjZY0Cuic2NyQqC1oPyuMcpgVdVtQyqosbzcyLQWheDzag25bA6FVCLMrxVB0FCVrdAj6TKhYM5zB72+1oCNh5c00ZhH2I4pT/p7LzeykBhOtDOcQrwm09GGxZPO7/ZJd/szXzliqzfNWtQnXPKW+SUCvHNIeHG2LfEF3TDiwKozAqgGcrx/SZDuJ/YcomwZSnAh0RDCSAgd2w3P3SoRjaJqBQ0XZxVU9Qp8u38qP8oSbBX+0wnXIaPu1M4TV7lyxH0h0lmV4Q7O3o7BXY9mGnipzu9Y3wUbnsKAoZinRCd8WLslc67HV4O+thOj8Oz6+0WOUarH1aenUSPfK9LAXR50X5tpFAK10VXMFnZnYm3gNNtuG/UP5uLviYvR9yY9PSkediHcSAbnW6pbIHMNysQbTbzPCfSkXm9iJQ9umUcpFeDUmjYZj44d+IMjYrvlLYYwOTBmG6az1lhryMmGK93ZtdEEjAT51oxmBhNOn+2FZnHSPNaqScw1NuWt4IGPRAVk2ekHOeD70SgtNPkraq0PhJAe7X5Lx2KCi6sLV+fD0fHzk+FcC6p2I6d5o0kL7MK8F0UYcAXMy6yQMuUBHggJuXSm+Ufb7gXEUODcqpwli0slMEIevJetSbf9grY/UJMsIbi7HCoGTZJy4J+F9BJCNWNkLzibF3B710BvcGTA0gAr3VpILvLE0Fu59vss6vScyKffNfSKhjjbZDUa22ZDssw+GkMTSU7LpApI6SBvgTcFvWOuXJ55iFOdUPWZs/HjAlWJUnkgui0D2lMHbYHMwbMppSJdm7J72IlCO+b2IoQIicQU4ZMi1K9uASY+5jzruxCrwjZi8H3hy4nxZUgs2UqP6mhItVZkf4ouoGiqxxNx8Vr+ILEthmHEAXuAVN/fIPwDr41EacuxobBnT7Flyo30VLPBnoROVqzfrZ8oqV9c7mCUjhDWdKCQDVg8qb6L9xX0VVXtZSVYegcfBLtxre9iiRG8nB5V0QHWZpnJmjZO4iXNb4FfswCS/xtaZPuSLchfV4Kif7xI5eZcMPPtAC72jzxxGed2vDRvwJlaNTSqNMG2vHifdUQnDUyVY9squN0EzYctALkAsY16S8uymqRJCmk2xSKM4gM4vF++ncobdINcntpxc6/bjD/J4S/zmUWMZuwI2badj+W8iahVhKT+qore9/55W8I+6gBjcgcLECqZFEgpGHNzUXtgRT2A97z7yVxM3Q10k1wHb+YgmL+qrltRQDKdhBeaE9FHp8/QXW/XiABcYcly4tq84tpx+LmBAAqtha1otnzYAzviN4EM0EyeSaeooOs04VfTnKOYWzZ6+GXCsg9ozmU1sUVoF01Utx6RHOmWOEJROf+9ZfTShWbr2wclcPXFjWGBfnJX7GjtoymgUWexNMAe6Y4V863rev94pRdVmYWFI7c6f5v9Dtblx6kjlcyd1/lBUryFm5xSDPjNbhNu4TCH6CeKXydt/mS+qzPmFmV6hbg5WKn3+sUEbcnf67PTS/pxV/TTtiNOu/H2AA8d13qYzBJ9X9Is9Y9uRQd7yVyGfzVJP65MlP1duBp45FK8DM3akgSNfHB7waW4qtl0lgFWFOvcl2LgLPcKgZ5juhy/fKHUE35zLpOIt1S8JiucFDYLZRKauodqyofi4hKcj9EhIHy8X8MSD0NY17yrY7safaHm6P9hPxB5c++/J9YnDpG4ngDk1DN69dMPP08Lb7CWzLtnOxb/V3jlqsl42gSFNvmkN2lbhbUL38f+bf8zc872SbxoQMTrWYVeFSrpEttM93+ZYhRyufCSRLpslXT1rWEyUYv26zYlOT8d2ih1nphtsayxHysxUBmpncNJkaYosrvgR31BwfVoGIrmoV1inTsxZ3z1J/jkKPKs7nCQNdGP9ew0xKAcgul0GYTLBxqM6P5jOIBPASInxEyLT8TqymJEcebidnA194RKm727I6HjSfl9410CEHGoVMb4oGyTF9vqKerhXNePY9p4ER6+SxvILf5okpXVC5mbVponSrBz1iZzVN9icQW+SrageYFKquaMeumgdtAOOzdwHrQZcnbqdrJ1N0DbWZsTDI4DUJti4Pt9v9Rhm1x85CSxi33z8wO4lpIp4cI40NjMEzA0voK7WAbXfDv2xPQrQIJo7luk8OX7n9nWY6ZaTt2eyrhX6VSAjYMzmjXjPO3oqwzbpEgiz6vtTPYrE18KLKIGPagd26UOmt78ssEDL32xvXoURm4sZNkSBlnoauCY6ychOjGq28EWE1GiW5xWt8I7i/QYp5oNaWBafJOeDiV2oowqN72ndrOJbW9CrLPBn38i/JBJzlFPkb4ovKfXIR30LYEIOXOKslszt4UZtRpUOu68Gm25Lt+8IXxyzvu7UUld73ceyifXBpvWHLaur5s3P+16I9EtZixVvxMkIoi4qaDXyazs7POeNpnztLVK4TiL84+akFHi/WlKM39jsb5jicoLnDOZ6ljZjO7FJ/OcwhPSuU+r/vYgR2oN1bN/KaInQZy5L09jy8vhRIwa5qvkWWAuP4Uzdk667sWmQAz++2S2W68O6SiYGEbs2NgQO4J/le7fRwCS3L6CPXrkGLq9T7VvLVClw91OnRq1TDX9iG+38c3Iw0nIbC+39i6fNuje+aXpOI8nYI5/s1mixEEJ04iDDN3iDtylIpImI+cSrYXNq8hojij58ivUsBJstTc+gwZZ9myPq1JwR7dkvk693ujr8iJqUtiAzdmHyn8+tuKITJqMVURsEFWJZHUHjrTb59kR99gBOyEBhi35IBHjKdHxbmzF/3FsE+HCRmzDJVYrGQ00nUUxkv6GUlMHkgxKHDtCSmpZlEndIImKvMYFQJoS57MVGxe9cU9LRBT3+ap+2cm885QOb5T5QEGNFqoEaHWDubvGMfvHgzHktrtWRotgbYVflXilX2ziExmYva7uB3P6b3K8Q0vBaA111YR5vTgNN26p+/ecD04i5orWqmZIwnx9ce2NRbPcZMS2f31Izy/hc6yoviv6WOgS95Cn3jAK70JQQOiYmoiwjbbO5tOIDr6SQfjUJ9xw3VTVyhgcaTgJecTVWvOMLoVq+PhPp1tPyjzd32310MX8EswvbWIMNkMLeVxp25Z/iDYrwjLbROrNHa4eQIivGg0QMRlYfJAn/L3oj1/UeUzsURxJAH5YgP+jjgLJuIg8uRtrGU2URlCzPeFzjhTSc1bIUsJcr0WnJ3arfBxT5af++ghYc1gFRTyvH77/69SkjQCsfMv2M+RYXCUEft2nIq/qt00VO1D++d+s+ahW43caIJElM6JXPFnuwAqbjO8qVs5vcNlKZqXNfRN0i6oR4gpT+DLs4q0QMr8IOtIbnKOkKygB16CQAJ5cLQWrqzruClkRl/w7Xop+PzBIN6goj6fShB+OAKQLrcl0ncQckur+P4yXaaiE0YjFOoFQsexWa9/TypF1sZoxApKJgius5ha6parMPTScfV1jfytCbz+t39kbBW7GWppmrT8mzsIw0U5Q6T3kZcaCCw+IGz3VmmfMo7C2FXftG91iBlvxeIHnwyM/lfQyL4ZhfqVICrFaqGOVmz89QokkQNakCijxHQb+RRS+gWNZaYE7bB7o1vFp9zWwMAWm74yynd2+P1pybm5Ybsy1rGjznrdYuiTVsNhX2Bzevldk8K5CshEJbYm7fPz2PDlNieq1kGcOp80xZC6iTJAp3aETLkLlbROe1FtD7n3FCe6smebXvH6s+A2ksGqxfVyC9USl24sq9A1R3avMLdfJbyLqza0t7RQ8+yqOhNxuQj8LFUEDEUz12Wv3fmh+2FaDSsLMCZW6G0/AVwxkpdof2YbhpEWNs281MQytr+haE3rQqCO91AzIp8nOX9qDZLTrlly0rbRRZazTJqZO6iCUk3HrnlbWlIgJd9Wski84/IBtJzyC9OGIOtJzTsILPGfSL7YfeJkqYmzR/E6nzGJnZ8A1Nd9BmzGKisF1e6Omo9AjDWeIPlGwqui10F6/zrQM65/bck4I6CDCgIPRJ0pYlDhDIcxbhPuu3BPdtF7EMqhSBthlvo2qvxSQLaWhyB/Si0/2bQpphtV5CUkz83zAwzJyxFdi3ZEgtkSt8FamXEu7rL19GKg5Jr+3eXMnR6Cr2x27lR6WpaevHvoJ5p7zm39Lk/sq97CCMf4I3oeeWF0FmRAzAmlrS3aihDB68mqkNs8uWWd4xPSjYOIl0J4RazLGSCrW4cUDLQGzwrgchjqi0wndzgLet63NrzMnEcehnj61yyapuu560eYUqbPqz4qIws0bG/UOmIpyv/szX2rOebGBVfTarmbELn9mArw/UWg8m+vaFDjaMaFNz5jVIcEhSRf8YqC5CVyvJ+NH/u+izDhJZjFFfBRcHOC3HwAVO6Pj/jlHKouaafUEB92fKNTNIjHcj5uYMKqiCj81CaugrpYpfGTwXZWOPY1eaDGgQ+s4kuA7RfsACTbXb3cKlgZZV2GbbAvCXuumSVInKuK/FIf2LAEheH/PEpU4hMlyoWMTl8JzA8ZSk13q5NMu63ngIFXQ+Xh15hJrNmTMpk1TtPQpzGYmUFiyuSZK8x5OIGGfWNXtJvZDewAXEC1ojzmMsCG0tvNiHYGLY8UATIxVW7CU/kderR0J2wAP34FtQRG1f1EToNeXLWVqftyXK/Wh/WxvH9wAirU/8yi5FQiyHXzdaYkr9Xal21P7oxBaDONqV0FTAhWH6Q3xJ6cx8Pg3LVmHImH5D57obgboiLhTA0ST0iGLFDQb7xWoF0J1Izef7uhFVm/nXOPM8XR3Tx8uDwJNa7c74BhtbvigIUUGO3uGRQdkgpKA10Ul7AblQE+JifJT3/dZrgAWvhnt88sZVkTd5ZRgbN5XMHEL7m8w9/w7tZqX6jNFuL/AFxIx9v5q74hKZddMAjRyJtvaOiAsGIcT6MQ2/kFCw/wykjilCnMWgFwAmEIhEgll5WSt0WXT5vS3T+lL04YcWcekah4y8uDAHDbwfPUKVbhbo3FOPIY+7IoXebaYO8b/a8GA3T0WTS5lL6lpWF7cqVjSpbA1sKM8/rMjKCgQHyUiCSzw55fFlmQadr6mZEuaPnemREjeIDjT0hqiIkjiyXlVOzPy0oTWixroQ7wjl/IcHw+tpBAA7YqQG2Na2xkPgV2nscHtmQ0k+bcFOSSrpI/FkKlatH+OxvETB/uG59zT/ssXF4BRbfhuv5UQ9Ja4zOMcGlmeUt3eyw0PWm+AUq/UP1FRd5EeRceh+tKhNXyurHSWES5uU8E7eFI+nPxZhO25YBqtl+Shk+CR3+lwcFcmbkB1SXcEyMOohqjIbtvXLa5bT4ThIhckGxxb227fJz35zs1oAUaZaY1tNS8pC6leC8fXRoNWpiLsWaYbckPIiU3BQeYCSqxeTn1476Fq28SDGnWuNjV4eq5Kglb5BiWIuPH2Vurpisg3iwFG615R+FzOBiDIRXnyBX9r/BpXFpTw2eVp1A8XCw81qUNoRp1GVy7KgBzh55tA6VEakSSuYGgSIz9VSdP1RGzkossR6QVko5gUJMhFydGn/3ofkdvzyVsORnExgxyVRRl5d2b60OhADA8ReDwl47iFLf6HGHkEgWvzWXohI0RcVLuhrHEE+E0iC5viRqLNRp5DqfZB56qaFCVfxBhJna1/H9ROkPxm5Ld2JhiMKjysOClqxfIjMO8aN1E0K9oHgsaK+KADq0Ilj58fG5AX/uMsm53elvLzhpfU7lN0ZM4JYKIrPk1CZlq29KjNrrYIAnF2iQgTK6jszC1YWgr7zT4h10KYZD4mfoldrQCpB8HFFe7CEHUts1Gty3zaaY8tPKe0blGypbXjA/chwxpLeDBdmpCD0Xm3i1EOKVV8BxDlgiYXBGuiRg+j674aRVUvE2ox98Y6a9uOsqrbELaoGXaQ76Lns+MJoe4d2JujXhIcXedN8ES/PZf4A6i5uVpOyHbki4aNIpI1UcXIepPMZFzFYuoWWTsgTKcwugNFvGrixt9hEwmnpqk2bv6IivwOrrawWyK0cIMFGGW+JPVdWpDOIXIO43OlSzixYZtD/AWXq3+BEltLX0UQ4wa9//6cF4tns+Bs8OSP4n9WB+sgNmZ+E2QyBcJLATFj54sP+3FgGQwfS3bH69vNnPfhO5AYsPHBm3jO/Wp0p5iBM9aLITgon/JEz8qfZoCpNTZtl/GtdiXCFC8baskd/NRSMzxp1yGRgtajqHJ6JsCa5VcMMu3stHptLKKjK7swl29YArwlPRnavNuOLTAGzO7bzn0T+EEVoEx+o7kOCuiQ7070NdpabOLfz5elHHjrDbZ0XTyZ/rD5FntvZSBFOqPgwOllRpYKv+41/YcQ6uHEKgGOWbMoBxAtZ27mN0Y3m04ofMOnTR5eumZOsXf3pDTDUYIXrrH64gH2QrmTaQIrFRyZNUvxNe4IjICqlI8uIkKXqG68EirsoY7px3atXfEkpuVDgGaUNavG9E5QZAf5UDGSVpFwiqA+u9MMTMC+rrPtuN6E74W3bTvHcdC4PFgVvXfCEY1B45f80ih0rdVjYMBmEsWHLIKvDQ116y9Ibx74QyfSSZeqkUl1fxycQvHwO+xsAW6mrmZWIlShtmOrUIMIui+jNGEbbQRNvjSafMZRU/Ny5TjFqdEMkUKcfpwIu+C1/2ydtKV44YF001LG0wNoa8pncrfwtW52FBVQQvOvJAh4Diru/M+FQAv7pqrMdjUKIUu7H6x/p0ZSWcMFzKqhYW849b3i5/JzJQJOsFgZWAls8EHkzFE9gpaVtY63fAhrV0MekqSXk6e3MR+cO8P3Ms9m2SDXsLZKAlSsGduuCLDsPppcDp7FhQc4zIv6feLE2MhUo8ngBqx8evSJJ18JYYLy8ZnA0sZHpVUqFPblM4u0TvI05JuAE7cLTqlNAtRaUErSjTsYPpGl7OKlLobzktLTgRsxBQBH00gxIZXNYeuw9/fTbwOA0hBzD+KsjIs0MZMdrJ6mo5UTXL+qZXToO2N0Q/uUnJtuFOQrH9BPMtwLxaUx6piByGTV+9MdkdbF6n6xUDkHH20ynDixy0JH6ShZM1imydW+VriL88W1u9YM8vNQJMevVAHjdOEBtr/nNxiGdsWOri+IcnDQtFB6Ny/YBV4RDKuxHTmoN8NGzbfdi77MnIHBgzu5ipFPaLnzwrOdeNNb8zkOE3NuBEVs4sFcCBsPbYOlszwdklyxkrEpO2QahlzR0dSxc1myeHAW9uKsrH+5Mcf0Lu77F0dJZRAzyu4gpAv8H/nocn1DaYYsAfulg8JU1u0w8uoyfb5zfHZWDyibP+P/CjEDvo+qYyBgGZTt7t2aBVa8bHA4OeUMb55xzfPE8jll5dIsO0jZcOZig/v8gXzlfLcxTOJ28rAnLzyYcPAsbyX+7L6ZO12RwnqsTjOvz2/UyfKzihYi5BkgdeKLNFJc1MdEf6EN5v2FNSABi7xMJktSvLshWFwQZrW3ts9VAHEG+gmjRzD/Zs0UtEKR8x3hw99InXHHEFlbzvYHzqctGqMU8ClILpYGUTPU3rTZdGv2HESibXxY1d92g4+DXRr3yhMaUkaKIyOvVBjene70kMz2rPafmkt8hG10hCZdMIBwQ2f4Hkeauic1DXkUerasAlIXnoGBdqEuNPur7Q16jDdO51YdX1lQ7P80UzhZj1J5lWj2YsIldehEzQPls7cdeu/KntmtOBrOSghd+SuHo7UeM1GFHEDvz1UhJsGDVBXhm9t4mM7yu8h5dO218cYBEkx8idCarHyUO8WqawgvmHqYVaFaTuAwizhs7w46aw4gFSoFXoSUr1nJQD/T44T1A/igzH2E+xmodyq1UfII1fyFnuFQ512eAdNWNpTX6aw2Cv0DvsK/ycrZ+SOPMescj5bYf5LUiOqQX+TxnzYBJcF/OyfnUdrULcciUrNsf0LmxPpUEyzDDC5Li8sd3ojcL8Jj7OMKUba4nptZm1Z01uNp4WhQ0XaaU+H910CLYoNPl6M9nT1/2gJrKHlNthq3unhSaTffRrTTpCSLuWsri+9qvrD80tOjAZGCjvAMy3R7QPwOfvMqyRNOozGWIsm/LOoPEObK8XKW4jVze0dWaBC+CXRAaUkin6t1/tnDysAae5a0dksbvc9Ift34VlflF1nk6TvPRVclDUUJh/Mj7GPJ/nW55A9HtTXvykuqFnFA/GnHKdhTj1JtKYb6V+Lkr80sLlqy/wa2zC6llRr4uxL9TwQ96QLIIvb8ZYeF5r9OD4+cuqrG+QM/sIU23AMmGyy7zx8lDWKE05SfTpN9iz/wjLCO+fmyE2iosjUKfNq5vWzmdZQAg4h6SmeDpNAVYoctT6xX7y+BGbbxfNwqL7FdQAANT2ExWY3kFjAAG0aorzAgAGE9yWscRn+wIAAAAABFla'))
assert hashlib.sha256(_v29_source).hexdigest() == 'bdb162dbc822622d34e9cdc0c4f9e91ee89fd7e11c33ad3113b4b6adccb5f3e1'
_v29 = _types.ModuleType('v31_frozen_v29')
_v29.__dict__['legacy'] = _v27
_v29_source = _v29_source.replace(b'import scripts.v27_notebook_core as legacy', b'')
exec(compile(_v29_source, 'frozen-v29', 'exec'), _v29.__dict__)
del _v29_source
_v30_source = lzma.decompress(base64.b64decode('/Td6WFoAAATm1rRGAgAhARwAAAAQz1jM4IdsKDBdABFoGYZjAex9znHhCdX19YWeU9j/POYt2DYePemSv+9QuF5cM9+qTQJKXBVdgyrWpAs9DT9cuKejOJ907f90GeMRpUDozRgrFWm5VGsEootOVjLNh+6GV6ancyt/hBvq/MpBGLeVEdAVyJVpESJncOEKsrz8QgGBxo7AF5moK7uxIeSHb0wm+KFNk9L4rGWfPTDQfl8n0h9WnArMkPy70w0acRu96JuUCKiMDHR0TdpDP64h1RZ4GoxfRy79khyQ9cazCR1B4nBh1FXt0c/uu6zifj5ghkJyIOunmgJDAGflOtJzAs2A8rq+amQmy6oCkV3qW/lqgfuGCm3nST5Q6fBRCxwjmn9VzzXPgflvMSiG0/eDTzPlT2uuW8nFVtRdPCMnrNv/2S34AslOSLyf8eKed9O2Mssf1U3ac9WslgZOHKRj30dcigGTncfW+BK17K8KYGnXTjQfBRyL7xjIwdQu2CR5v/q6sD7mhfDXGXbrzkvoBVn8VU8OoGx2HfrfvxkSS6zBCd0hw/kXCkr6Rma+IyxKfrWPOj/tz6AluSHhSgQ1v9c5VMH7FYYNx3zKiJ4ZhQlhL8bqPuzb93IMtvOHVeWBqnlHG2VQbnFBS4uFEBJPa1eNsZNOC8ip60WKUVxvmh0k61YCMbQAIlxvPttMHJ8JNT5iiKwkxtQRuhtSW8aDOixx9WiVVbnEYdlBjt1jkh6JVxFwsIZ2yx6fSTnz/S8dIumx76yLr5zFSgWlry/r+Vve6TzidSZQgU9H2Y+Gq7Ig8E3Im0hQ22PPXFOrxq5DPZ81pHQ5JT/tp9zZF68xnrItEbEnxv6lPN2JsoEO3XebduLIxW68TQARrBXgMigAUUIDfDpFTAhNGN9uFQOYveV/D2m8gC4rXQnRd8ImOH55pV8zH7yjr02ZHkyj3JngcGlhQQtGJcGEPP+HXclXOZBM2ui+K1SYBJHde5j+SqvNRAJVJc8UXLtz5376RM9QOtYAuUuRkl/220TAQzk2rdDExPWjISMF0edUnCw/ATa1XAiLTfR2cQuV3X/cM1OeaxOmwoghfbXErBMB0hXhXKoQJHYl5z+/KMI4zT7kjPRzqwwzcWUh4BIuFN8Y9tXGZHggEp0tThJqr84qEuk8elihkW5daBwePPub6IHJfDVdRGSsuPMv9pKVMz3RYqIjUE1raAzf0q7AhRQ/aFXpJNp32RsM7nXwXEUme9554jPSAAQp4AKA+cyKGzqbOc3EIbqbsJ59fiJRxMG2sq+rrtUxA8STZ0I4h6Mf5dMes1PqonEwuE1IgjlAHzC/DHkuAW/bakx52RzqVyEI+Zvq+lI+hJXdLnar83lZB546QE1DS49S41XsISbgWcZygktRjffL3csqZZ6PJuBPyf8pXL2/RmSTmKtKu8msp8tixr4E+qijJejm2qVi68u8zybCQl6AQQKHy022mTxGqIq5Y79VPDdKkWHCTN51SBrM7q/MfIIa4dmnyS+H5MZxJ4R1EEVrrCzpzDIPK6jaEub1s8uhvy7jgbnZS+yKGWnb40RnhTnnb3F3TazhBNdgFJKsbpKZZdnKhGDANTR7cJdup1qHBgsGBgoEiwF7Y522rNEmYYfQMClOhGiuCbYEkA38kaGPm1Aiuit6iG3VqA6KsCpj8G3ax2azfHZFvFuJRZKS/oj/NHpy/pnwUaM0s4CAzV25EY2mE0MArREzxwGdq/uRsesk49ipEF5ctvJmU/Qj8WEp9QPfmFPGxN+S5aqbhXuCahKmafLAUq36SGhCoVxoKUOnNyMasWmK5wYcZOQ7V4K31LuUJINLBoflTCuQRh6YrtZdp8QdVg44PW9lpZJd57bBA3POZbJspXyaGibgczE8ig+6VChFZuJHSb40qlt3XfTeCrsjsyqspNDjRNoCQbb1D4BE9XjFY60XGgfIPc0xv09ssPd51461mYQVKjNtQDWfSHwQD4l/GGa6QP1wyFCxVJ/5uNeDsvu6PrfC9dCLL0/JWbehGcptiPxNFojOLdomiUZJxr9BGtoFDAGGcOAcie1RIZUlC9NzSHApbL6SAFzJ2zIMHEOVJZphW0qyLsOKQNNJinEG+oNbZ0L4+ZKODZH/luBd7VxQEfiGg3kwMkfvgt3NC6dcrBbsBlpNSzW0ZuI7W3lavc88Ky+FMQHA5HvSwMXOBt2DURWW2+j+EGr/S02T6pA92scDXRt/vkCIK9Bp3AHlo0I5kHaUGsAPJc1axXLPgG1cvD7ErOglRymde+H+Rc/83UrbnX3qwRRGTmO9hiczUMghyhgsN7ws4HtzebW93bitaCtO7MbgfNb08KozLXN5f2Ye5mKMbqKWANwbCmwxcJayGH2p/o+Y2ZNRZxtwsBl7v3Egoew1E3M84KlCovr4z8nfOa/j0x2h0CcxMH8hZexFmaFK2d3p2/szCCo7Umlo1MHUCkCjcjXrbunLctVFHFDDr0ETVVYTYEpaw/Lk8ZOxJIPg/Z3EzGh0IeDJ1IgVdeKGA6SV3cigzHPWoMaENGD4VTfpasr4G3lFEdp6/R/tT5L4O296+ZGq6R3aOeKoFvXeO4TLnkZw3+Tq4YSmk0iNMognJk3gabLsue9wdPl5NTSnKOxSTDydM7LHiiAvYAaHSx13ABXphrhnUDd/1vlPfOKN7lyxJAHD2RtjCw1H2fqvfgYDSHwbMSGNl3wfs2SR/ZTDUVqc7UA74leDwFsF5+ISugyL7TETlN1viJKAH+6sfHjvLfoCdJ7J8IvsCf6st8R3WT87Fq+cwiCVfoWemenb8giWo3xG40V2hN7uoT9HpPAi7oSlTx7ZmoWpnrHrWDmd0W4mMZrVQPWpyN79vLPNbkCi3JHtEaldtbn6Q79Ia1IeRAT+gCMfihtsDsOvYJPqd/RCC+2mZuuML7SM3l5A5hw3pgA1fJKDtnLzdkAGyQ6hUVwXgbp3CHgwvcWOuYZO94q4AiKsWBcwC0A8NKPRqfsl/zE9j/c1KjDdd51PVx4leNHSfwGWHBv4VFlBlaRpsx8OAfdBdAfPUMfgI407igBn8ZHp66voZarKwctuzYn40+NMVZOHT+0KjcLWbDLX8CMKQMUktfhCSfI76CaUB0KzCp5DnAkjlnxQRds3dh+rLWQa97wTzieCT3GkFUcfjLHa9uFsQD0fDNZ9n4JvPDU/ZS5GyRYrxfeWOUdhnMbI8783tStEzpiAEBBmFAd1c7o0VfUONroslHHKkDndIc9sHQ6Kkdx0tMnwGKGihkdjGynznQqlX8D/yd2q+UaohBqnwV9U6L8MsjfDGqukeKc24ohjdmoEIrJA58U1f39OUuwGsi1iqE11ZHAc8bNSPu6rM18b4HrD9Gx1JkJ6MUGVlIDAYHNaqYWeAOElCzvOyvjHBpnNG3F6ngvMIOSNrBv+KyQPaHSKC+eu7EtLkl4vOhIqlpBkvpcVRRN+48nG3SMGZ0D/syGL1FLqBgPxFJ9TXwO8Argaq1C/r86MYVdkWKGKuQikDMDUtpzqDM2fc5T/C8ykkbj3YgMkp+v5hCtJgm/KRL2zy8943/ff+cREOoBxm/cgTdZXZzWzgY0nbPvDXL28gYHe/Kx3ux6/ptwMg2zefslr36zLnR3/EB57PfxETM+Rr7LvHeCq/xLDlPFeiXXa9YY+YYnRHhIA5m+gJwpGEe+4pMmfAYr5lz/9k/iSuXdApdzUnSeKaiCV3NKtZDg6oxgsphYJ3lk5G1iBbpffYSs2bb+fGc8WmaK8Q5joj9+STUbaVfFrHjTdRmUHmW/pjX/lNXsNifvR8NdDuKpsSHQk1TgEvRX6aICoBn3RZBUwRlQIHUSJeiPyFAL6fVqei/b2XLOhQ5YDQpF5okpw/+R5KSuWsKi8ItPDl2rOhIcWghbiaC2JZXh6Y32P/jhuqctyseOPGwSBN01DljDbcdESXdRi7BeML7J+i06+RgJrmrXexKJ9b3B6Hm5KMJ4RHRB7NNP1O00qPfoqTJ8eX4BnmWB1Zp5BaEldb10xPwwC9cBZluLRCqzPEw/G2FZJdkFm6/rJCUF6NXysRu75pibOFj0p/pb7KmdwytGqC8GM9O3MAMmEFYyV31LgycTEUh+MLgaKz3EFupMegnFrMHVLHztjb0mEb/jQNJjFTMBJwZqLbDN5JdjOUwujUioFc+Hn+zzZEbCvKK0gXp6nftcSHy4++wPLlNTWhVGjgM9XYfedgsiXPxe+d2JHXH0F5ksGyXfG7tqjVCVmXNhWoksUNak0xhXAmmjOJvJxn0DdpV3qjYx3pLxyXhvAmbaDFX/TSbbXnXCgBpfAL3wfeV2b2s10Xn9noApRJmavS2uJNCtOaurmxoYhYrLsT96r1iA8a4leoXu8RixqnLUoH3c8cpyJuQpFOEKB1NkuKeKSyw9QpYli2kr2I5eNbTUmkuxkH7rPDM7f3yElHwmbbAXnQ4TPMFWysb2JHVfmkdIvf3daVmmLspd+Y+gYIel4K67EIjvOGEbOPFtPOSIVqE9L5nEowV347rVfIyK1y+GeIzspt0tznXoXT/yddGAu3tnv22Iws1Zd6krc/Lt3iWPmJBOGxe0vm41mG6jx09/Wv7piv8HO7LidrH1pSYOrp7lKilnNccO2Mrw//n8kNVG8kMSBm1eMLcryxpn4Cpkr0ZkRhp7Ek7Bc3TRN/HdoWODgA38GaSU+8FpGewYQuN0Tpn4uFwng++FdJFoT1z4nNYQW20KzA+4bCV0P+Zh7/rgVvIy1BFj369OQkRfDB4rKoCTLnUsJNgJnC7tS3iEWJHi65g+40FuY46snHdlwbZ12IaT2JFi98S+sGaHya2mkNAH78c5lqoN+tnBLVg6U6Bz49c86upqst09piVXZL1dSCT6p/alriTTvp5YPoUshQUXUIuNkgDVWf03y6rX/cekKaoID8XtCm4pB0IQJ6gSMTq+wqJIKXA8wVfYA9NZ1DhhqcREgFtmkpR6geATnymeUtki5lpddftWwnb55wufM6NCBQmFVpGvidYhv9kskb4IthbRP7j0X7WMavOhg+058bOMbN1WtrV+mJ4eEFrjvMMy2f1+vNsXGPFZAi3fO+XLpEmfqStC952UP9kTkt4wDvyade738EyZRNIXvTy4dvHhdEKRzOcRf8nehJPXGq1YhjwEjppkHaAMtj8WpEk2AUsiumE1+keMr2/e0QcyZU8Fzd5uLy+Ba5ovjEY/UNPIv/UuWPeLvf6WyLy3v0KpuT4qR86VEEB9GPeImNKM76MMxs/XUOdj1xCEF9bq464Z3yyklAPfxlkjFfBOK4jW9AqxkLtkVUoFNZFPLJ2Q9/THSPUpKxU48T6957MEB4IkkNKw0fpjHe1gQkRlbHMfDfbHjggzlCytiDta0NWogD9geL50ZTPgSJw1TmtGl9iNANDaZhBeEt6fRSo7Q5lPi0FWiDZ+zFk4u7Pahg4A79MUPAo/HfxBz+cqeQE/IeOZOTlUHd3fEFJ8jhL3JSiz/IYSujSDhysc8RhM+yR+e88eDIZBkRIXt81Lus1UfPQrjIeTe4DEMYa3K9QOOWCPn+58XrVQFJ2E9+vzYTQnOlff54BeczU+6NEQohfDMUHj4OCxRALxG3LKxQ3HekrUaZANQxZVHQvSkx0WwzrLbyorr7X6k1KUfoXiuWuHAj0NYrj8FKNkyDVyBWt/CJ+Ys18spO0DE/tvyvvmjYTueT47xP2DpRcWQ2S6WODHvSPnWc6cL8F0FdokWwzi5bdZVz/lDiSq8uLenUua4oZyMn4l2kUI2elsl82CoHicGXkpVlR1IRYSNIb2JY7Vugyai13cTj1BhXD3GUol+jGUTPdR2AcyOVJVKev7gIW7zWI3K3UzUK+7gRgedIK5NJPfQlmnZr3R18ZHG7TXvDRX5g1CZW2JCN06Nxz9SGBbY5awSTquURfnrzHB7Pp5R14pIzNimGE/Wd2Zeyxmi9lLoYX/CHUI4NnkljCLyQxrXvQxDnhVDxMmSurJZQdtRwp7PwT4sxumz5T+kQRMSgoglH0HqqDeNEg6tEBdWdA5KrbdHi4/qmfvqxQHrLNCWgenhAV/Lk30Cawef8AYVWCOO+xRq79eMNYzSsUfVw0eY1NZaccKkcrrLGSUum8Sp8dhtjdCkFiUCQuDTU9SVWkfMdncKzZNVVtuq+yMHs0AXSPDWYvOPo052HE3sTJDGDu9DIxbGwiDWNY0N1HXavZpVAUbxIfvMY0H7QbNP23SAGSv/UsDi+tgAVGKxT3C/CEJ05i5IvRJDZXt8wD/7p5GiBmhXgyQBA2D2hJmf2iatMW/Wrl5IV8ktGMG47hoH9+A1YJRP6OPyk5n4QpCr1qSxiZZXsOOB4xYgPwucbMdT0PuErh0UbQgg7JxKQJSCBPvbAQIeRToopxNc6ZCjw25gyRT8IgWCBGhjWsQOdW2kudNJNtmJWA9pRgBlacYp+A6aW6kn9G96UohqQxBN+Mp3Is5/v4Hh76GWOns28KhkvldpSjWkylylA91jJLoPsbtao/f/0mzqHt+McV3Z2zqpXfeqZaiwiGUtVc++Udi98wMdxsliFDUMPEfDwgzh83KUcQCKThZRznJCYsqWXOr+FbBSobAYAUIYnltOnnJYVRtj4nuMAygEpAmP9c8jiV6Ef/6pgkORbmdiCLL9PE0nXDYHl0n5sjPlIk1tIozeUfvGSyFOLmPqBrfWGZXJ7kNVZkIqL8vYA8XvSnqJxnQDo6qIIjbpi3fh6f7Jyn7r035eERG7oyQNIbyC835akwbvIyQdLzs618dHXceV+jZG0sbWR5+wXVBvFyXVTiRiSI2zDELfYJRUh9cORYHeI+FpdVaXOdFIMEcJ2d/XcLYnjpZbvMRRGMZDcv3/AjpZCpMvUCo7k91HcASqSbCfiUSSWi4D8gltvoHrlYujNmUIjVHU6D0IIF5J524HdyQbyiAkiufOr13gUeb30OkW4Yh2SmxXpoMS9cJxt2Osm3CPwYYU3wQlNgw4u7YGweV6ZvvfMRvW1iaqINfxbaFNHheUzNznOG4FHPB8xWUZNAHIv1v+CWqBvZruE3SJFTgy4YvaMfwHGDaMvucwEAiyD8Uat3Z2Rekdby0V//tzSSVAfYAe2F0yAmLX7WyB+1cbF4TQaTTHpv1EdtZ73PG15cM5BjgRk8mu5MaXehUMT8Ro8wdr81AxrkzY6aRs0qLR2YQxWFTVZdaySSqp5SmbQAyvxzJyUxI1xBWZy/eNYpkcHX7qHxj+qLMEmf645ejFSjvkJVhF0ZxKF6A1VG9iCjybt7RC58/G2zqsEhHG6E0HAZ5b0heZPDMYB3TbWRqg7WDT+aOaGryZzICNOjZrGAbzzhwwAnJcsO6MgfSQd5VXbA9XkXXvHs071HQrMoqxK0O50JsaFBqT6Wf/LzdNGvDvxrucuYvWOuqWZhlT/NPNGWBV4PRllCfqMCIHTIECEtrp5Pso+PM2GoTfjJ+VsHoc9UZkUS7I1DNgo3NiDz1GOzf3BHOUklV27cvZP05oztj238+JpbwDixa6vPmDjY+eXwwv/IQcRkHyBhOea5mU6wbLa19iFX05vr6PPZ0gpf7XeiRVCrAGfk+wtcjyKNCpVR1LnP/Plnroz0UE2R75CEabOk5GhvBF7OdTip/6TKORk8nrj/GE+jXUch9yEYtuZm6Kz3sciw9xAb3sv2gkr95fTIayn8hHfNg9uGQcW62f9u31t/jehgDj/q1Mnn49U0vqBOSAOOKYg9vUHgxI6szRBHlL7QT5jX6IdseM3LHu0OamjI4azcV1Hs6E+7PqunRdZK4D5VBf+ApiMCnpmPR0CgYHGr7DDvKordHnOtoWFww5brpwPS60YhNc9ofJL5g2bxtABksJ65f6x5BpZ7CisFMgYtWt0zrYSza+hKQJG5ovAUPAOX0DEkVHHQhHNCc8QHcLaN5a0g8VNBMB9q8lWsqYK3NrY+HC6cZMfB6B5UGqUIVRHaGrvr2CcVA/BL4mL01DcX2CEuDYDJBVOkXAY6Uzm6zoSWoYTqstw88mNQcZPO08jXejBZAhCABguogWi0po2BukR/Ux5Q2+YBY1HQOfCj1JiNp5B/JCUwrezCiYsCS8MrQlSy2JnsruEFgqmUA4+NF1l8K9in6KTNjY/nOuAp9wolyh1PbwTAQE4/D3jvPCeDywhwkioLj0IndFeGxptfKKI/zZhe0VBhvu+qcpqnLwlLyTmttggoiDbf1HKFTiHf+v0tD9rOj0cpKyz9gFAsTJX1bKfB/50ooYBAVyZHma15mjECZJftlaQKMBh/s3syXGz2Wpu9u9lJiZ4T3aTivW/ae58h/J/2qz89Nv5KnxohPmDClH5sHIhAQMQw43fPDnQk9zhBvS0XNsEMRS1K4wM+C1q4OKm3Lr3lNtJFDpoXpiDkN5IL50qomyMtELZ+BVeEVuq8gMDdtBqV/r76eIAHDsGE60Pry+N6t3pXl4a2KyVn3yz2atkKwn/TqrmUNX9xrn5mDMNEcpJ5986rFDGe5S1AqDncOGHLQ190PIwp4HZWlu7ynbiDLvRW729RNPEO4hJkAxsVpAvRuEboHS0r+qlweVUyMb9AJeXoH2zSUOsPlHSZHjgHUmzRs3GNjyjNDitXk8QTtYXVGB/UdBInNuhpRprOKAtgSKIxwyeR/MRyNZauzCWB7NerYKlXCKhjTDU3mj8fNBT82r4cNC62TjqKiQxTs09N7QtEE6SRIw+awCeglyeCO7AMo4ZOY7mpNee4VBmFBo5SrKun6jU2BFEKHjp2zAgz4uGC/Q9rKez6imIdOJT4OZCN6cQBtqzLvioi7FdTDEZA8H2Q7UopHGj/2vRPBAbLhJItWBq+VzelCwb3RKBKuZ9E2R8ehtoUdjKN7ajcdpeX00qIlZ8TQh3pCSiT4ilMQT/NK07E+7EiHiZPVV3beauw3Xh/UxKQNK4VlRNXd37mFTtex1vU9r2Bz+PYqtfXJ9ReMzaweiTfguUhh3keIYXfj5u9ZTRJ2CzJRCTku7uFBJhiwJIaOrjjGsMudCh46Lj/o51UxgOmJkMmOup/3Bb5G2HEC5KxqcgvXrio6milJwEL427ppG9zAMf3Fc86bTYrQz1HKEi6s4R1kbowrkBtKLOgxj7RrAu0H1PdBBH3h2zRLzEWUQUPdKotHd9Zyj+C6AGzO4mmIL49E+uTxu52fq7fSz2IaZfG83YIXcIazyZnBgNFTh5KF5zYoCvfGpOtxtv95Q2SKP8tQ1W2h/S5jq8dlTy4OZ2Vb55NF/S1dA0L1+OgfiFiPQbhkZf+4maO82UO4loy71eD3PTcKb6RktRy/fBpb2y9sild1QVLCsbLnSIJ2VnEFAJv3a5dlcwHqsCvMKqJiYui4aUEyMyHVj9RCuGDMylxZ+v/h4kSdrftUuzGAKSHXblo5AssyqxKDpmHbefxJB9mqoE7iaH50oDH0JfYvRV7kV4zuaj1NK37swUm9VsWD5YWoP5E8LywpYejdyQKjyg+8QP7MLuB2vqZrp3CfkzeUwAiqloMN1HWKqKhPyCOGyEl8MH68DmoE8RZZA+ZgJmIGLJsRCivDuPfwCMrLRjiBzNnAeqkoUzBh5A5UI0Q+dII53yoqIzGuBiLWG0GR2bkHgp76IsNkAJq0/VZYR8CLC/onUudBR2wRivnKdFTxnfk+D/uedKsd2/QPDVI1UlG5BNahhEwmhykcb6nrvelvpygMXLkb86ZPYEdXjwsNgHgJjU4CGoLKPzzwAyxkVPdnDV3lGgRlSe4JnaerUqsAkcQa5MLX3917ClFP4C/CAbbuC9NqERIWaN/Gn3/Qpphq6OodcssQBNzsFGqpaGBcBgxkMpyIQjU2ucl2rJWF+Tb3/YVdvnzAMKdZu70CVMESBL32Q6gFxVaQYVOYBkLJSAaQmYbZlT8kXV12t+0O3IuDBwrdqwxzlV1DuobsqohO1jt9R4Art+6CVhaQ0LuLOFaU65q10A+z1C0AJq+Sd03T/VFo5TyXLR9u8HSMQ9UzicWOvoPHaRu8Dw26JrmSG5+EWDVRDWe5oyyGNoebG7NwQ1hz5vw+4M34GurJs5qKR+gFk0U236auOA9US4NBwgoBbLA+l5q/Bnoohcyenc0RthLOaMpNfzHWts2xMxTtYXktUdcLson5GluOf+azqwyScf9jJpu+ZtQXssTt8/j0t8035j2ojvbnzq4ePX7aU5I+FQLvzzPS5zMPJeUDosKWByr9gxgwEdmzBntvEt4wPYAwKGMbUfSvyrYDvfRaZvtl0R1+9tNt6Uz7yp4lnMLihjJmPntBwVtLppr+XCbx588Sqv8E/0VCpu794c195uS2HTmeSV5WSM2C2WXx6kXfEHenL5PN7vom4/rpdg15ZIVd/ctVy9O1kgELDUa+5LSakfJifP+GtLPybwPfnpZamzNBe7RBUKqUOWh6ksbM5x/t4mfBtTMSCkOCsZq8m/U79qnotFBeHEyKTepb9SlYGQsr6XfDeyyO5ahWHK9bhw47a205lqrYz5/oGoKr4VoivgRs6Yg7bQYxYlsW+31ZCWqt9qIDnqwOJ0mrOAZVFUnmv08nU0uIcmVTZfgQQ8GCHTr0+QSX3bATGYlBhAyY0gmpfhG7Y6qMa/9Z9V6Ut3Xcf5vX7OYg2xRb1llEGLJD/TtzJJcQFtAXpr1rvSp+fTK+xBueJf6nf+md49dkEHydSB6xKW8yMvyfeXB2fa71f+PBTeuAWCi6QyIRqLAVr8ZD8KrNQacK9oiMz17GIv1l4WTY/j+Gxr2PQw0ShQ4Xvc2Ni7iz6jL6fd0OoXyPSVdnJ17lfngDfhhlRMLKZguUvArtnaU9fzTFhOQquIAzUW6gI10GwKaOcOmEf0p9p+dAhTrHv5qltrhONeG52orM3nPe9cN5HIW/ZifRmLhvHjkk/Djk124qnp+k/ENu2dUVmWL6uJ32xxr7nfoafBN8f7aWv5uO8X/2L2Pm5ptL9caYrKHUPWu3BDfFWhEa/3j+6LxMpB7WIwkUbcQulQIZmmtzz3ozYaB1Wcv2BoRiAG1SJdLdOpVthh5ftvur15HzOAeQOH9mIONp0gQFCZ9ax3/4kjhQSQiHagam8+FolP8TUNROwAgEh01+R8EL4Fi3n893ZjZ2gKYXxb8oIt7TG0Cds+LfdNELKiFaRK5qwex4wo3ZklSGzol1ge0QKZcqZJWSTMKaSlnI0nVgahBD/5Ufj6+YOqoF5lJy5wvVAWoicW/vx0Yo151sHTc/HY7/vWePgj9Qg/F05Dz3SpHV9D+LruSKKDJXFrnzoH2EQwt8Jw3X70/PdiCSKIPF1jjXG+UgpL9qYWVpy7mLRF1x5c8P8ivcjQyE2+QZZUvl3Q0I5cswGD6uHYQNVVKMB5PEh286/vA2yoyrPL6XTp8JDKqrR4gTlebJJVAcG4JbJssZbqUyegUGQK3rpz8FQNREefP77U1oPp8M7cQCn+IeyqkOP7hJqicUUqXJwzDGPg7Fv1DSaZ7msEnBd4T5MtHkO1VgdIx98h1rvzMLETEPNnDddexFwW3I5t21Uaq9hG1HJUPAkNFU0TybrmblXLX0ZYGCA9AxuQap3dI4EeNtRzLKB80DSbs1l/+CjzsxjoeDb4KAWUtcw2LPICJ0mFmG66RM/mKETO9g/hdMdI8f5S8KHM6wdNwTcPhqeKikjEmxz0O427ih1f+yH9+YIrZI/nbQLQ6k2YodzsVHC71JpoN4O/U3nshUIppDE9Arn1royWN9occLXr4s/YL9mtRUM9Yceu8OLLef76CFUxPCPMYxSqTauVrbsaDIdxZg+gHZwcmflAnGje3oR5wN0sqVAXdzvC4vtH4YvCykIzia3UxtyRdvRUNMT2040f5ctndM76msq5RsRtNAdQZuCXKTUzfnITNr2wF47DDkhxGEGethSDKJ4dlU7/iDloaMmm+bi0wop72JNdcyimJHztlUu8RdvANHLOgqpPh6vvonGLnriWzCs/qDpJeDQmKWHgDOuiVKQjUM/R7sBZWMIMq6JOE004+5IvcqNpKIwoWAVjOB58oSlFGvr3epU+3Mb/OZiFZkW4SXeVXgMPJivK6xFb5gT63z96W+D893LgXTV5i3xJNoBPvtpFa1f8JYAkvcClC9mYfFngmRBv43TtyaFiS2/lhqm03xZFP55oSFCp8/e/V34koz+HxIjv/TE60UpOCB//w8sbCmp5YMKx+zmy5zvafAw+gyzN7eVLrgKShH7gUyVOgFMkBvshjovFOHCeQBOrQBRT4frmBKv5yR2M/CZfp2fG2wXq58YxD/IPg4T7OPClGAA4EzEw/gmf8IBYyVZDSuNOTbtNii0LBNQJ14rdxkv4FWWAYq7IHKz1lWgGfYht8WJADVr18HiE5Kyg+M0Uxd4zFaOeA0cN4wKYpzcoceDLMERe1CEovjCb36BAarw7DqYKVEIA4/KlSclLFQBK/CtwOZ7Q6sZfExdPMtV3igdAMzNzxoaJ2JEa31nl3Uk2owaB/uNua/t784SuJYhz4XkAjUVuxbWaGCjt/6dtezDiOerEBch3Wm3EB0+cu8hbPeS6d5PSVt4ZX0hqwMVakkXU1lwBnieHwx9fssL53Gl5olWMp8NzJMwB5WyexVTN3qilmlKSpLUlMnxPVcGCGeV/aEGtWipxl3NNG+qSRaxM0OTGXZTVKhME2IeYkhjALKjKDpcCVVdr1+0nC8EKQtu6yHnTOf3HsbXjnzV5FdDW18Ck/GP6FlyJYs9gqpclAvdcBPgrHPGNxa4CwRZsX8kCIVMkgzSWBfVOQxZc9UPGnnG7VRPA8JIQ3agbBUnpxn84N6kYSgPMlKlJYnma+nsHNP/v5dKqfOO/UPVJeDtJq+Znspfz4ancWSPFsmBM6D4cJf7ckl09k9A08vLn40OFKgEF5vo3o5/WSKIJ5gA1eYHerGeG+Ex09jMOvZ8sEJePprj4qQoQuC9SzApviXsIRdpeTd62ZB21fVIWoSAFgwM2Qv8lrD647LeuQjWpVpoSvkZrN3FkdVnf5P7/rox6DXTZb6g4VkBD1mtIe2n5DizlRXKvv+OaR89NkEpSsIXxKCMy8ACKfJCnekYuxNH4VE/EL8w/RCIZeRWcLuzLFzkSEY0/xuUO9sltbEhQ/oXvppUUUCWitOV0C4LkZmXl2ToZE9gUmVfgZTW/8TN5JGaKKUk6qI1PtPgmMkxwmWJPsJgi9HkMLnup2m+KUwaYSjyvC6hZBxDTeQt5c6HCg3cUHaLruoQB3XjPHNur3JVKfxDOLdguz2cR1vN4+aMF7Iiy0G6fFD/mnWBlOC1Zfn/18qQzLsPL9DsKC0z5vY2tiaIa59pXNty24iSvLCi4tNBOsI1SrM/8m7uVOliGiwq6LuUs5Iheja/LKTQ4LxfqeeIvQySUuSR7GdwZk+xlEp2tlGSUAkUrwkGrKvW4jv5SY3+Uh4ukmgNBZkMU2u3dYvpCI2gu4qFjHcpoy1UR276EBrZlept0rY8ON0OSCEF73HpCI10GvN+RZswFb93OLld5QcYGJO/M06s8WYTrlZPY1TA/cesWa4C3X/QUQRN12Kg9nfOvQY6QoV1zH9QE7qVNOe5eK7Wxug1tJFv1mA4NrkY/fzAlOwGJ+gWeu4xD2OP0ig3UmgPUZUAh1Vh68oQXoYcn5xMJZVJDU5cxbEMGrtU7tdMu72f9AlwFbf7FE2mQKnF8TkRszFoVvkA60tFlyEAZVMAAcxQ7Y4CADQI5GCxxGf7AgAAAAAEWVo='))
assert hashlib.sha256(_v30_source).hexdigest() == 'faf8f4d9b106097a5de7f11a156f4ea99cee809815970f2d802ae7f72634bc03'
_v30 = _types.ModuleType('v31_frozen_v30')
_v30.__dict__['previous'] = _v29
_v30_source = _v30_source.replace(b'import scripts.v29_notebook_core as previous', b'')
exec(compile(_v30_source, 'frozen-v30', 'exec'), _v30.__dict__)
del _v30_source
previous = _v30
del _v27, _v29, _v30


v29 = previous.previous
legacy = previous.legacy
EXPERIMENT = "v31_full_data_bagged_habitat_residual"
CONTROL_HASH = previous.CONTROL_HASH
MAX_HOURS = 10.75
CONFIGS = tuple(dict(c) for c in v29.CONFIGS)
# Freeze the epoch schedule of the best *scored* production recipe, not v30.
EPOCHS = (12, 15, 18)
PRODUCTION_CALIBRATIONS = (
    {"slope": .931340859404444, "intercept": -.031887795053823956},
    {"slope": .8515955628102694, "intercept": -.23047593294208438},
    {"slope": 1.9269219246980456, "intercept": -.5758844842834113})
POLICIES = ({"id": "control", "neural": 0., "habitat": 0., "swaps": 0},) + tuple(
    {"id": f"seed{n:g}_habitat{h:g}_swap{k}", "neural": n, "habitat": h, "swaps": k}
    for n, h in [(n, h) for n in (.5, 1.) for h in (0., .25, .5)] + [(0., 1.)]
    for k in (2, 4))
CORE_COUNTRIES = ("Denmark", "Netherlands")
TOP_K = 128


def compact_rank(probability):
    rank, values = legacy.top_rank(probability, min(TOP_K, probability.shape[1]))
    return rank.astype(np.uint16), values.astype(np.float16)


class HabitatAnalogues:
    """Train-only environmental PCA and exact nearest-neighbour label averaging.

    Country, survey IDs and coordinates are intentionally absent from the metric.
    Query labels are never an input. Sparse multiplication avoids B x K x species RAM.
    """
    def fit(self, environment, labels, indices):
        self.indices = np.asarray(indices, np.int64).copy()
        if len(self.indices) < 2 or len(np.unique(self.indices)) != len(self.indices):
            raise ValueError("Habitat fitting requires distinct training rows")
        x = np.asarray(environment[self.indices], np.float32).copy()
        finite = np.isfinite(x)
        self.mean = np.where(finite, x, 0).sum(0) / np.maximum(finite.sum(0), 1)
        x = np.where(finite, x, self.mean).astype(np.float32)
        self.scale = np.maximum(x.std(0), .01)
        x = np.clip((x-self.mean)/self.scale, -6, 6).astype(np.float32)
        components = min(32, x.shape[1], len(x)-1)
        self.pca = PCA(n_components=components, svd_solver="randomized", random_state=20263101)
        # Handle an entirely constant fixture/feature collection without NaN variance.
        if not np.any(x):
            self.pca = None
            self.divisor = np.ones(components, np.float32)
            self.latent = x[:, :components]
        else:
            self.pca.fit(x)
            self.divisor = np.maximum(self.pca.explained_variance_, .05)**.25
            self.latent = (self.pca.transform(x)/self.divisor).astype(np.float32)
        self.labels = sparse.csr_matrix(np.asarray(labels[self.indices], np.uint8)).astype(np.float32)
        if not np.isfinite(self.latent).all():
            raise FloatingPointError("Nonfinite habitat training metric")
        return self

    def transform(self, environment):
        x = np.asarray(environment, np.float32)
        x = np.where(np.isfinite(x), x, self.mean)
        x = np.clip((x-self.mean)/self.scale, -6, 6).astype(np.float32)
        return (x[:, :len(self.divisor)] if self.pca is None else
                self.pca.transform(x)/self.divisor).astype(np.float32)

    @staticmethod
    def neighbour_weights(distance):
        small = min(32, distance.shape[1])
        temperature = np.maximum(np.median(distance[:, :small], axis=1), 1e-4)
        weights = np.exp(-(distance-distance[:, :1])/temperature[:, None])
        local = weights[:, :small] / np.maximum(weights[:, :small].sum(1, keepdims=True), 1e-12)
        weights /= np.maximum(weights.sum(1, keepdims=True), 1e-12)
        weights *= .5
        weights[:, :small] += .5*local
        return weights.astype(np.float32)

    @torch.no_grad()
    def predict_rank(self, environment, device, guard=None):
        latent = torch.as_tensor(self.latent, device=device)
        squared = latent.square().sum(1)[None]
        count = min(128, len(self.indices))
        ranks, values, distances = [], [], []
        batch = 128 if device.type == "cuda" else 16
        # TF32 can perturb very close neighbours; float32 matmul is intentional.
        for start in range(0, len(environment), batch):
            if guard is not None:
                guard.require(20*60, "habitat analogue inference")
            q = torch.as_tensor(self.transform(environment[start:start+batch]), device=device)
            d = (q.square().sum(1)[:, None] + squared - 2*q@latent.T).clamp_min_(0)
            nearest, indices = torch.topk(d, count, largest=False, sorted=True)
            indices, nearest = indices.cpu().numpy(), nearest.cpu().numpy()
            weights = self.neighbour_weights(nearest)
            assignment = sparse.csr_matrix((weights.ravel(), indices.ravel(),
                np.arange(0, (len(q)+1)*count, count)), shape=(len(q), len(self.indices)))
            probability = (assignment @ self.labels).toarray()
            rank, value = compact_rank(probability)
            ranks.append(rank); values.append(value); distances.append(nearest[:, 0])
        return {"rank": np.concatenate(ranks), "value": np.concatenate(values),
                "distance": np.concatenate(distances)}


def residual_decode(base, neural, habitat, policy, guard=None):
    """Protect the first 60%, preserve EVERY row's count, change at most 2/4 tail items."""
    if policy["swaps"] == 0:
        return [list(row) for row in base]
    result = []
    for i, old in enumerate(base):
        if guard is not None and i % 1024 == 0:
            guard.require(10*60, "bounded residual decoding")
        old_set = set(old)
        protected = set(old[:math.ceil(.6*len(old))])
        scores = {int(s): 1/(10+j) for j, s in enumerate(old)}
        for expert, weight in ((neural, policy["neural"]), (habitat, policy["habitat"])):
            if not weight:
                continue
            for j, (s, value) in enumerate(zip(expert["rank"][i], expert["value"][i])):
                if value > 0:  # Never promote zero-evidence habitat species.
                    scores[int(s)] = scores.get(int(s), 0.) + weight/(10+j)
        tail = sorted((s for s in old if s not in protected), key=lambda s: (scores[s], -old.index(s)))
        outside = sorted((s for s in scores if s not in old_set), key=lambda s: (-scores[s], s))
        removed, added = set(), []
        for outgoing, incoming in zip(tail[:policy["swaps"]], outside[:policy["swaps"]]):
            if scores[incoming] <= scores[outgoing] + 1e-12:
                break
            removed.add(outgoing); added.append(incoming)
        result.append([s for s in old if s not in removed] + added)
    validate_residual(base, result, policy)
    return result


def validate_residual(base, result, policy):
    if len(base) != len(result):
        raise ValueError("Residual row count changed")
    for old, new in zip(base, result):
        if (len(new) != len(old) or len(new) != len(set(new)) or
            not set(old[:math.ceil(.6*len(old))]).issubset(new) or
            len(set(new)-set(old)) > policy["swaps"]):
            raise ValueError("Residual count/prefix/swap contract broken")


def geographic_gain(delta, countries, ids=None):
    frame = pd.DataFrame({"country": np.asarray(countries).astype(str), "delta": delta})
    # Both calibration folds query the same survey IDs. Count each survey once
    # for minimum-support country checks; predictions still come from separate fits.
    if ids is not None:
        frame["surveyId"] = np.asarray(ids)
        frame = frame.groupby(["surveyId", "country"], as_index=False).delta.mean()
    country = frame.groupby("country").delta.agg(["mean", "size"])
    supported = country[country["size"] >= 30]
    outside = frame.loc[~frame.country.isin(CORE_COUNTRIES), "delta"]
    macro = float(supported["mean"].mean()) if len(supported) else 0.
    transfer = float(outside.mean()) if len(outside) else 0.
    return {"gain": float(frame.delta.mean()), "country_macro_gain_min30": macro,
        "outside_core_gain": transfer, "outside_core_rows": len(outside),
        "supported_countries": len(supported),
        "robust_gain": float(.5*frame.delta.mean()+.25*macro+.25*transfer),
        "geographic_gain_positive": bool(frame.delta.mean() > 0 and macro > 0 and
                                          transfer > 0 and len(outside) >= 30)}


def fit_fold(number, split, rows, store, temporary, guard, device):
    directory = temporary / f"fold_{number}"
    directory.mkdir()
    stats = v29.fit_normalization(store, split["training"])
    records, calibrations, role_data = [], [], {r: {} for r in ("calibration", "assessment")}
    for group in ("reference", "replica"):
        sums = {r: np.zeros((len(split[r]), len(store.species_ids)), np.float32) for r in role_data}
        for i, source in enumerate(CONFIGS):
            guard.require(3.5*3600, "start fixed-recipe development fit")
            config = {**source, "seed": source["seed"]+number*1000+(31000 if group == "replica" else 0),
                      "id": f"{group}_{source['id']}"}
            model, record = v29.train_candidate(store, rows, split["training"], split["selection"],
                stats, config, directory, guard, device, fixed_epochs=EPOCHS[i], phase="development")
            p = v29.predict(model, store, split["selection"], stats, device, config, views=2, guard=guard)
            cal = v29.fit_platt(p, np.asarray(store.labels[split["selection"]]))
            for role in role_data:
                p = v29.predict(model, store, split[role], stats, device, config, views=2, guard=guard)
                sums[role] += v29.calibrated(p, cal)/len(CONFIGS)
            records.append({**record, "group": group, "member": i}); calibrations.append(cal)
            del model, p
            gc.collect()
            if device.type == "cuda":
                torch.cuda.empty_cache()
        for role, p in sums.items():
            if group == "reference":
                role_data[role]["base"] = v29.decode_ensemble([[] for _ in p], p, {"alpha": 1., "count_scale": .8})
            else:
                rank, value = compact_rank(p)
                role_data[role]["neural"] = {"rank": rank, "value": value}
        del sums, p
    expert = HabitatAnalogues().fit(store.train["environment"], store.labels, split["training"])
    for role in role_data:
        role_data[role]["habitat"] = expert.predict_rank(store.train["environment"][split[role]], device, guard)
    trials = {}
    cal_data = role_data["calibration"]
    target = np.asarray(store.labels[split["calibration"]])
    for policy in POLICIES:
        prediction = residual_decode(cal_data["base"], cal_data["neural"], cal_data["habitat"], policy, guard)
        trials[policy["id"]] = legacy.score_prediction_lists(target, prediction)
    guard.stamp("v31_fold_complete", fold=number)
    return {"number": number, "split": split, "data": role_data, "records": records,
            "calibrations": calibrations, "trials": trials}


def select_policy(bundles, rows):
    trials = []
    take = np.concatenate([b["split"]["calibration"] for b in bundles])
    reference = np.concatenate([b["trials"]["control"] for b in bundles])
    for policy in POLICIES:
        scores = np.concatenate([b["trials"][policy["id"]] for b in bundles])
        gains = [float((b["trials"][policy["id"]]-b["trials"]["control"]).mean()) for b in bundles]
        geo = geographic_gain(scores-reference, rows.iloc[take].country, rows.iloc[take].surveyId)
        allowed = policy["id"] == "control" or (min(gains) > 0 and geo["geographic_gain_positive"])
        trials.append({**policy, **geo, "fold_gains": gains, "allowed": bool(allowed),
                       "sample_f1": float(scores.mean())})
    chosen = max((t for t in trials if t["allowed"]), key=lambda t: (t["robust_gain"], -t["swaps"], -t["neural"]-t["habitat"]))
    return next(dict(p) for p in POLICIES if p["id"] == chosen["id"]), trials


def regression_check(bundles, policy, rows, store, guard=None):
    frames, summaries = [], []
    for b in bundles:
        take, data = b["split"]["assessment"], b["data"]["assessment"]
        target = np.asarray(store.labels[take])
        base = data["base"]
        prediction = residual_decode(base, data["neural"], data["habitat"], policy, guard)
        old, new = [legacy.score_prediction_lists(target, x) for x in (base, prediction)]
        ablations = {}
        for expert in ("neural", "habitat"):
            ablated = residual_decode(base, data["neural"], data["habitat"], {**policy, expert: 0.}, guard)
            ablations[f"without_{expert}_gain"] = float((legacy.score_prediction_lists(target, ablated)-old).mean())
        frames.append(pd.DataFrame({"surveyId": rows.iloc[take].surveyId.to_numpy(), "fold": b["number"],
            "country": rows.iloc[take].country.to_numpy(), "spatial_block": legacy.spatial_blocks(rows.iloc[take]),
            "matched_v29_f1": old, "v31_f1": new, "delta_f1": new-old,
            "predicted_cardinality": list(map(len, prediction)),
            "swaps": [len(set(a)-set(c)) for a, c in zip(prediction, base)]}))
        summaries.append({"fold": b["number"], "gain": float((new-old).mean()), "ablations_not_used_for_selection": ablations,
            "multilabel": v29.multilabel_summary(target, prediction),
            "species_groups": legacy.species_group_metrics(target, prediction, legacy._frequency(store.labels, b["split"]["training"]))})
    frame = pd.concat(frames, ignore_index=True)
    if not frame.surveyId.is_unique:
        raise ValueError("Repeated regression survey")
    return frame, {**geographic_gain(frame.delta_f1, frame.country), "folds": summaries,
        "surveys": len(frame), "matched_v29_f1": float(frame.matched_v29_f1.mean()), "v31_f1": float(frame.v31_f1.mean()),
        "bootstrap": legacy.paired_block_bootstrap(frame.delta_f1.to_numpy(), frame.spatial_block.to_numpy(), iterations=1000, seed=20263107),
        "fresh_assessment": False, "used_for_policy_selection_in_this_run": False,
        "by_country": legacy.summarize_by_group(frame, "country", ("matched_v29_f1", "v31_f1", "delta_f1"))}


def production_estimate(bundles, full_rows):
    return sum(max(np.median([h["seconds"] for h in r["history"]])/len(b["split"]["training"])
        for b in bundles for r in b["records"] if r["group"] == "replica" and r["member"] == i)
        *full_rows*EPOCHS[i]*1.4 for i in range(len(CONFIGS))) + 45*60


def fit_production(bundles, policy, rows, test_rows, store, control, directory, guard, device):
    take = np.arange(len(rows))
    records = []
    neural = habitat = None
    if policy["neural"]:
        guard.require(production_estimate(bundles, len(rows)), "whole full-data replica ensemble admission")
        stats = v29.fit_normalization(store, take)
        probability = np.zeros((len(test_rows), len(store.species_ids)), np.float32)
        for i, source in enumerate(CONFIGS):
            config = {**source, "seed": source["seed"]+31000, "id": f"replica_{source['id']}"}
            model, record = v29.train_candidate(store, rows, take, np.array([], np.int64), stats,
                config, directory, guard, device, fixed_epochs=EPOCHS[i], phase="production")
            p = v29.predict(model, store, np.arange(len(test_rows)), stats, device, config, test=True, views=2, guard=guard)
            probability += v29.calibrated(p, PRODUCTION_CALIBRATIONS[i])/len(CONFIGS)
            records.append({**record, "calibration": PRODUCTION_CALIBRATIONS[i],
                            "calibration_source": "frozen scored v29; transfer between seeds is an explicit assumption"})
            del model, p
            gc.collect()
            if device.type == "cuda":
                torch.cuda.empty_cache()
        rank, value = compact_rank(probability)
        neural = {"rank": rank, "value": value}
    if policy["habitat"]:
        guard.require(30*60, "full-data habitat fit admission")
        expert = HabitatAnalogues().fit(store.train["environment"], store.labels, take)
        habitat = expert.predict_rank(store.test["environment"], device, guard)
    prediction = residual_decode(control, neural, habitat, policy, guard)
    swaps = np.array([len(set(a)-set(b)) for a, b in zip(prediction, control)])
    return prediction, records, {"training_rows": len(take), "all_PA_rows_used": True,
        "calibration_anchor_rows_removed": 0, "cardinality_equal_to_scored_v29_per_row": True,
        "production_epochs_frozen_from_scored_v29": EPOCHS, "changed_rows": int((swaps>0).sum()),
        "mean_swaps": float(swaps.mean()), "maximum_swaps": int(swaps.max()),
        "habitat_metric": "official environmental values only; train-only PCA; no geography or IDs"}


def decode_control(payload, template, species):
    # Keep the frozen decoder's globals isolated from notebook source globals.
    previous.CONTROL_PAYLOAD_HASH, previous.CONTROL_RAW_HASH = CONTROL_PAYLOAD_HASH, CONTROL_RAW_HASH
    return previous.decode_control(payload, template, species)


def publish(export, template, ids, predictions, species, gate):
    tentative = export / "candidate_DO_NOT_SUBMIT.csv"
    proof = legacy.write_submission(tentative, template, ids, predictions, species)
    differs = proof["sha256"] != CONTROL_HASH
    eligible = bool(gate and differs)
    name = "GLC25_PA_submission_v31.csv" if eligible else (tentative.name if differs else "unchanged_v29_DO_NOT_SUBMIT.csv")
    if name != tentative.name:
        tentative.replace(export/name)
    return proof, {"eligible_for_submission": eligible, "different_from_v29": differs, "prediction_file": name,
        "message": "SUBMIT ONLY THIS CSV" if eligible else "DO NOT SUBMIT THIS OUTPUT; keep the scored v29"}


def save_compact(path, bundles, rows, store, per_fold=1500):
    ids, folds, countries, bases, ranks, values, distances, truth = [], [], [], [], [], [], [], []
    for b in bundles:
        take = b["split"]["calibration"]
        positions = np.sort(np.random.default_rng(20263108+b["number"]).choice(len(take), min(per_fold, len(take)),
            replace=False, p=v29.sample_weights(rows, take)))
        selected, data = take[positions], b["data"]["calibration"]
        base = np.full((len(positions), 40), 65535, np.uint16)
        for j, pos in enumerate(positions):
            base[j, :len(data["base"][pos])] = data["base"][pos]
        bases.append(base); ids.append(rows.iloc[selected].surveyId.to_numpy()); folds.append(np.full(len(selected), b["number"], np.uint8))
        countries.append(rows.iloc[selected].country.fillna("unknown").to_numpy(dtype="U64"))
        ranks.append(np.stack([data[k]["rank"][positions] for k in ("neural", "habitat")], axis=1))
        values.append(np.stack([data[k]["value"][positions] for k in ("neural", "habitat")], axis=1))
        distances.append(data["habitat"]["distance"][positions])
        truth.extend(np.flatnonzero(store.labels[j]).astype(np.uint16) for j in selected)
    np.savez_compressed(path, survey_id=np.concatenate(ids), fold=np.concatenate(folds), country=np.concatenate(countries),
        reference_columns_padded65535=np.concatenate(bases), ranked_species_columns=np.concatenate(ranks),
        probability=np.concatenate(values), habitat_distance=np.concatenate(distances),
        true_species_columns=np.concatenate(truth), true_offsets=np.r_[0, np.cumsum(list(map(len, truth)))].astype(np.uint32),
        species_ids=store.species_ids, expert_ids=np.array(["seed_replica", "environmental_analogues"]),
        evidence_scope=np.array("sampled previously consumed calibration; truncated top128, not full probabilities or a fresh audit"))


def self_tests():
    assert len(POLICIES) == 15
    base = [list(range(10))]
    expert = {"rank": np.arange(30, 10, -1)[None], "value": np.ones((1, 20))}
    assert residual_decode(base, None, None, POLICIES[0]) == base
    result = residual_decode(base, expert, expert, {"neural": 1., "habitat": .5, "swaps": 2})
    validate_residual(base, result, {"swaps": 2})
    assert np.allclose(HabitatAnalogues.neighbour_weights(np.array([[0., 1., 2.]] )).sum(1), 1)
    assert not geographic_gain(np.r_[np.ones(100), -np.ones(30)], ["Denmark"]*100+["France"]*30)["geographic_gain_positive"]
    return {"passed": True, "tests": 5}


def run_v31(control_b64):
    guard = legacy.RuntimeGuard(MAX_HOURS)
    working = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("artifacts")
    temporary, export = working/"v31_runtime", working/"v31_export"
    legacy._clean_directory(temporary, working); legacy._clean_directory(export, working)
    temporary.mkdir(parents=True); export.mkdir(parents=True)
    store = None
    try:
        device = legacy.require_gpu()  # Fail before opening the large official predictors.
        torch.set_num_threads(min(os.cpu_count() or 2, 6))
        tests = self_tests()
        root = legacy.discover_data_root()
        preflight = pd.read_csv(root/"GLC25_PA_metadata_train.csv", usecols=["surveyId", "lat", "lon", "country"]).drop_duplicates("surveyId").reset_index(drop=True)
        _, manifests = previous.make_splits(preflight)
        guard.stamp("preflight", folds=manifests, fresh_assessment=False, epochs=EPOCHS)
        original_writer = legacy._write_remote_arrays
        try:
            legacy._write_remote_arrays = v29.write_multiresolution
            features = legacy.prepare_feature_store(root, temporary/"features", guard, workers=6)
        finally:
            legacy._write_remote_arrays = original_writer
        store = legacy.FeatureStore(temporary/"features")
        store.high_train = np.load(store.cache/"train_sentinel64.npy", mmap_mode="r")
        store.high_test = np.load(store.cache/"test_sentinel64.npy", mmap_mode="r")
        rows, test_rows, pairs = legacy.load_rows_and_pairs(root, store.train_ids, store.test_ids)
        del pairs, preflight
        store.static_train, store.static_test = v29.candidate_static(rows), v29.candidate_static(test_rows)
        store.eco_train, store.eco_test = v29.candidate_static(rows, False), v29.candidate_static(test_rows, False)
        splits, manifests = previous.make_splits(rows)
        template = pd.read_csv(root/"GLC25_SAMPLE_SUBMISSION.csv")
        control = decode_control(control_b64, template, store.species_ids)
        order = pd.Index(template.surveyId).get_indexer(store.test_ids)
        if (order < 0).any() or not template.surveyId.is_unique:
            raise ValueError("Test/template IDs mismatch")
        control = [control[i] for i in order]
        proof = legacy.write_submission(temporary/"control.csv", template, store.test_ids, control, store.species_ids)
        if proof["sha256"] != CONTROL_HASH:
            raise ValueError("Exact scored v29 round trip failed")
        bundles = [fit_fold(i, split, rows, store, temporary, guard, device) for i, split in enumerate(splits)]
        policy, trials = select_policy(bundles, rows)
        legacy.save_json(temporary/"frozen_policy.json", {"policy": policy, "trials": trials})
        guard.stamp("policy_frozen", policy=policy)
        frame, regression = regression_check(bundles, policy, rows, store, guard)
        gates = {"new_policy": policy["id"] != "control", "positive_each_fold": all(f["gain"] > 0 for f in regression["folds"]),
            "spatial_ci_positive": regression["bootstrap"]["ci95"][0] > 0,
            "geographic_transfer_positive": regression["geographic_gain_positive"],
            "geographic_buffers": min(m["minimum_distance_km"] for m in manifests) >= 20,
            "exact_scored_control": proof["sha256"] == CONTROL_HASH}
        prediction, fitted, diagnostics = control, [], {"skipped": "development gate failed"}
        if all(gates.values()):
            prediction, fitted, diagnostics = fit_production(bundles, policy, rows, test_rows, store, control, temporary, guard, device)
        validate_residual(control, prediction, policy)
        gates["within_budget"] = guard.elapsed_hours() < MAX_HOURS
        submission, decision = publish(export, template, store.test_ids, prediction, store.species_ids, all(gates.values()))
        frame.to_csv(export/"regression_per_survey_v31.csv", index=False)
        save_compact(export/"calibration_top128_v31.npz", bundles, rows, store)
        report = {"experiment": EXPERIMENT, "status": "complete", "runtime_hours": guard.elapsed_hours(),
            "official_submission_made": False, "official_public_score": None, "official_private_score": None,
            "control_scores": {"public": .23900, "private": .21093}, "private_target": .23021,
            "selected_policy": policy, "policy_trials": trials, "regression_check": regression,
            "submission_gate": {**gates, **decision}, "submission": submission, "self_tests": tests,
            "training": {"development": [{"fold": b["number"], "members": b["records"], "calibrations": b["calibrations"]} for b in bundles],
                         "production": fitted}, "production_diagnostics": diagnostics,
            "hardware": {"gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None, "torch": torch.__version__},
            "validation_status": "all 88,987 PA IDs previously assessed; repeated spatial development, no fresh holdout",
            "limitations": ["No guarantee of SOTA, leaderboard gain or full completion on an unmeasured GPU session.",
                "Geographic gates were added after examining v30: they are development choices, not independent audit evidence.",
                "Matched v29 is a fixed-epoch recipe refit, not the original model weights; only the test CSV is exact.",
                "Production uses frozen v29 probability calibration on new seeds; transfer is an assumption.",
                "Conservative tail swaps may reject useful but larger changes; all row counts intentionally stay at v29."]}
        legacy.save_json(export/"v31_report.json", report)
        manifest = {"experiment": EXPERIMENT, "source_sha256": V31_SOURCE_HASH, "embedded_v30_sha256": V30_SOURCE_HASH,
            "embedded_v29_sha256": V29_SOURCE_HASH, "embedded_legacy_sha256": LEGACY_SOURCE_HASH,
            "splits": manifests, "features": features, "configs": CONFIGS, "fixed_epochs": EPOCHS, "policies": POLICIES,
            "control_sha256": CONTROL_HASH, "runtime_cap_hours": MAX_HOURS, "no_external_data_or_weights": True,
            "fresh_assessment": False, "outputs": {p.name: legacy.sha256_file(p) for p in sorted(export.iterdir())}}
        legacy.save_json(export/"v31_manifest.json", manifest)
        size = sum(p.stat().st_size for p in export.iterdir())
        if len(list(export.iterdir())) != 5 or size > 16_000_000:
            raise ValueError("Compact five-file/16MB export contract exceeded")
        return {"status": "complete", **decision, "runtime_hours": guard.elapsed_hours(), "output_bytes": size,
            "export_directory": str(export), "regression_gain": regression["gain"], "fresh_assessment": False, "policy": policy}
    except Exception as error:
        # A failure after CSV writing must not leave a ready-to-submit filename.
        ready = export/"GLC25_PA_submission_v31.csv"
        if ready.exists():
            ready.replace(export/"failed_DO_NOT_SUBMIT.csv")
        legacy.save_json(export/"failure_report.json", {"experiment": EXPERIMENT, "status": "failed", "error": str(error),
            "traceback": traceback.format_exc(), "runtime_hours": guard.elapsed_hours(), "official_submission_made": False,
            "eligible_for_submission": False})
        raise
    finally:
        del store
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        legacy._clean_directory(temporary, working)


In [ ]:
V31_SOURCE_HASH = '35b414a5a804ab2eae663ae619d564c5014094d081e01531cec6f781bf934fdb'
V30_SOURCE_HASH = 'faf8f4d9b106097a5de7f11a156f4ea99cee809815970f2d802ae7f72634bc03'
V29_SOURCE_HASH = 'bdb162dbc822622d34e9cdc0c4f9e91ee89fd7e11c33ad3113b4b6adccb5f3e1'
LEGACY_SOURCE_HASH = '69a2d3952fa6324d9a05f4edb6fa7662388a198df89c407f7f034b94099f1dc0'
CONTROL_PAYLOAD_HASH = '0206bb2ba4587f03d84d8adae7cd5e9efb59f2893b90c234242743c3dc332e3d'
CONTROL_RAW_HASH = 'b425ae9730c2ec8e05b9ac889059bd58b96bfc10db21402f29effd4c4397720c'
CONTROL_B64 = '...'
print({'source_sha256': V31_SOURCE_HASH, 'control': 'exact scored v29', 'fresh_holdout_remaining': 0})


In [ ]:
V31_RESULT = run_v31(CONTROL_B64)
print(json.dumps(V31_RESULT, indent=2))
print(V31_RESULT['message'])
V31_RESULT
